# Data Trust Score v2 — Deploy Notebook

Runs the full v2 deploy against **`EDLE_DW_DB.PNC_DATA`** using the **Track B** (no account-grant) DMF path. Generated from `ddl/*.sql` — regenerate with `python build_deploy_notebook.py` after editing the DDL.

**Before running:** set the notebook role to `PNC_DEVELOPER_RL` and warehouse to `PNC_WH`, then **Run All**. Stored procedures run via `session.sql()` in Python cells (the notebook SQL-cell splitter can't parse `$$` bodies); tables, views, inserts, and CALLs use SQL cells.

Track A (native DMFs) needs `EXECUTE DATA METRIC FUNCTION ON ACCOUNT` + `SNOWFLAKE.DATA_METRIC_USER`; skip those cells unless the grants exist.

In [ ]:
# Procedures are created from Python cells via session.sql(...).collect() so their
# $$-quoted bodies reach Snowflake as a single statement (the SQL-cell splitter
# would otherwise break them at internal semicolons).
from snowflake.snowpark.context import get_active_session
session = get_active_session()

## 0 · Session context

Pins role `PNC_DEVELOPER_RL`, warehouse `PNC_WH`, and schema `EDLE_DW_DB.PNC_DATA`. If you deploy with a different role/warehouse, edit the `USE` lines here — the rest of the notebook inherits this session context.

In [ ]:
-------------------------------------------------------------------------------
-- ddl/00_setup.sql
-- v2 Data Trust Score -- session context.
--
-- Pins the worksheet so every v2 DDL deploys under the same role, warehouse,
-- database, and schema. v2 objects are DTS_-prefixed and live INSIDE the
-- existing EDLE_DW_DB.PNC_DATA schema (no new schema is created -- the deploy
-- role cannot CREATE SCHEMA on the shared EDLE databases, but it holds the
-- PNC_DATA_RWC database role, which lets it create DTS_ tables/views/procs and
-- the DTS_CLONE__* clone tables here).
--
-- Production tables are NEVER modified: DMFs attach only to the DTS_CLONE__*
-- copies created in ddl/20_dts_clone_and_dmf.sql.
--
-- If PNC_WH is not the warehouse you use, edit the USE WAREHOUSE line.
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

USE WAREHOUSE PNC_WH;

USE DATABASE  EDLE_DW_DB;

USE SCHEMA    EDLE_DW_DB.PNC_DATA;

-- Sanity check: confirm the active context before deploying.
SELECT CURRENT_ROLE()      AS active_role,
       CURRENT_DATABASE()  AS db,
       CURRENT_SCHEMA()    AS sch,
       CURRENT_WAREHOUSE() AS wh;

## 1 · Registry / metadata tables

In [ ]:
-------------------------------------------------------------------------------
-- ddl/01_dts_registries.sql
-- v2 Data Trust Score -- registry / metadata tables (governance-driven dims).
--
-- These tables hold the human/governance-curated signals that Cortex DQ does
-- NOT measure: dataset inventory, element (column) inventory + CDE flags,
-- business glossary, ownership, authoritative-source certification, lineage,
-- and classification. Scores for the registry-driven dimensions are derived
-- from these in ddl/30_dts_scoring_engine.sql.
--
-- Grain notes:
--   DTS_DATASET_REGISTRY  : one row per (report_family, layer) scored object.
--   DTS_DATA_ELEMENT      : one row per (dataset, column) -- element grain.
-- All tables CREATE OR REPLACE so the deploy is idempotent.
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

-------------------------------------------------------------------------------
-- 1. DTS_DATASET_REGISTRY -- the scored-object inventory.
--    One row per object we measure across the INT->DW->PUBL lineage.
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_DATASET_REGISTRY (
    DATASET_ID            NUMBER IDENTITY(1,1) PRIMARY KEY,
    REPORT_FAMILY         VARCHAR    NOT NULL,   -- POSITION_REPORT | TRENDED_REPORT
    LAYER                 VARCHAR    NOT NULL,   -- INT | DW | PUBL
    DATABASE_NAME         VARCHAR    NOT NULL,
    SCHEMA_NAME           VARCHAR    NOT NULL,
    OBJECT_NAME           VARCHAR    NOT NULL,
    OBJECT_TYPE           VARCHAR    NOT NULL,   -- TABLE | VIEW
    DATASET_FQN           VARCHAR    NOT NULL,   -- db.schema.object (source)
    CLONE_FQN             VARCHAR,               -- DTS_CLONE__... (set by clone SP)
    DQ_MEASUREMENT_LAYER  VARCHAR    NOT NULL,   -- BRONZE | SILVER | GOLD (confidence tier)
    IS_VIEW               BOOLEAN    NOT NULL DEFAULT FALSE,
    FRESHNESS_COLUMN      VARCHAR,               -- ranked #1 anchor for FRESHNESS DMF
    KEY_COLUMNS           ARRAY,                 -- composite key for DUPLICATE_COUNT
    MARKETPLACE_STATUS    VARCHAR    DEFAULT 'INTERNAL',
    IS_IN_SCOPE           BOOLEAN    NOT NULL DEFAULT TRUE,
    CLONE_EXISTS          BOOLEAN    NOT NULL DEFAULT FALSE,
    CREATED_AT            TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT UQ_DTS_DATASET UNIQUE (REPORT_FAMILY, LAYER)
);

COMMENT ON TABLE DTS_DATASET_REGISTRY IS
    'v2 scored-object inventory: one row per (report_family, layer) across INT/DW/PUBL. CLONE_FQN + CLONE_EXISTS maintained by the clone SP.';

-------------------------------------------------------------------------------
-- 2. DTS_DATA_ELEMENT -- element (column) inventory + CDE flags.
--    Element-grain scoring in ddl/30 rolls these up with the CDE 2x multiplier.
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_DATA_ELEMENT (
    ELEMENT_ID              NUMBER IDENTITY(1,1) PRIMARY KEY,
    DATASET_ID              NUMBER    NOT NULL,
    REPORT_FAMILY           VARCHAR   NOT NULL,
    LAYER                   VARCHAR   NOT NULL,
    COLUMN_NAME             VARCHAR   NOT NULL,
    IS_KEY                  BOOLEAN   NOT NULL DEFAULT FALSE,
    IS_CDE                  BOOLEAN   NOT NULL DEFAULT FALSE,  -- critical data element
    CRITICALITY_MULTIPLIER  FLOAT     NOT NULL DEFAULT 1.0,    -- 2.0 when IS_CDE
    PII_TAG                 VARCHAR,                            -- source WPII_/WSPII_ tag if any
    BUSINESS_TERM           VARCHAR,                            -- FK-ish to glossary term
    CREATED_AT              TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT UQ_DTS_ELEMENT UNIQUE (REPORT_FAMILY, LAYER, COLUMN_NAME)
);

COMMENT ON TABLE DTS_DATA_ELEMENT IS
    'Per-column inventory. IS_CDE + CRITICALITY_MULTIPLIER (2.0) drive the CDE-weighted rollup. CDE candidates seeded from repo PII tags.';

-------------------------------------------------------------------------------
-- 3. DTS_BUSINESS_GLOSSARY -- business definitions coverage (Definitions dim).
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_BUSINESS_GLOSSARY (
    TERM_ID          NUMBER IDENTITY(1,1) PRIMARY KEY,
    BUSINESS_TERM    VARCHAR NOT NULL,
    DEFINITION       VARCHAR,
    HAS_DEFINITION   BOOLEAN NOT NULL DEFAULT FALSE,
    STEWARD          VARCHAR,
    SOURCE_SYSTEM    VARCHAR DEFAULT 'COLLIBRA',
    CREATED_AT       TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT UQ_DTS_TERM UNIQUE (BUSINESS_TERM)
);

COMMENT ON TABLE DTS_BUSINESS_GLOSSARY IS
    'Business glossary terms; HAS_DEFINITION drives the Definitions dimension coverage %.';

-------------------------------------------------------------------------------
-- 4. DTS_OWNERSHIP_STEWARDSHIP_REGISTRY -- Ownership dimension (foundational).
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_OWNERSHIP_STEWARDSHIP_REGISTRY (
    DATASET_ID        NUMBER NOT NULL,
    REPORT_FAMILY     VARCHAR NOT NULL,
    LAYER             VARCHAR NOT NULL,
    DATA_OWNER        VARCHAR,
    DATA_STEWARD      VARCHAR,
    TECHNICAL_OWNER   VARCHAR,
    HAS_OWNER         BOOLEAN NOT NULL DEFAULT FALSE,
    HAS_STEWARD       BOOLEAN NOT NULL DEFAULT FALSE,
    CREATED_AT        TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT UQ_DTS_OWNER UNIQUE (REPORT_FAMILY, LAYER)
);

COMMENT ON TABLE DTS_OWNERSHIP_STEWARDSHIP_REGISTRY IS
    'Ownership/stewardship per scored object (foundational dim -- gap flag if unseeded).';

-------------------------------------------------------------------------------
-- 5. DTS_SOURCE_CERTIFICATION_REGISTRY -- Authoritative Source dim (foundational).
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_SOURCE_CERTIFICATION_REGISTRY (
    DATASET_ID          NUMBER NOT NULL,
    REPORT_FAMILY       VARCHAR NOT NULL,
    LAYER               VARCHAR NOT NULL,
    SOURCE_SYSTEM       VARCHAR,                 -- e.g. WORKDAY
    IS_AUTHORITATIVE    BOOLEAN NOT NULL DEFAULT FALSE,
    CERTIFIED_BY        VARCHAR,
    CERTIFIED_DATE      DATE,
    CREATED_AT          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT UQ_DTS_SRC UNIQUE (REPORT_FAMILY, LAYER)
);

COMMENT ON TABLE DTS_SOURCE_CERTIFICATION_REGISTRY IS
    'Authoritative-source certification per object (foundational dim).';

-------------------------------------------------------------------------------
-- 6. DTS_LINEAGE_REGISTRY -- Lineage dimension. Seeded with the two confirmed
--    chains; UPSTREAM/DOWNSTREAM null-ness drives lineage coverage.
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_LINEAGE_REGISTRY (
    LINEAGE_ID        NUMBER IDENTITY(1,1) PRIMARY KEY,
    REPORT_FAMILY     VARCHAR NOT NULL,
    LAYER             VARCHAR NOT NULL,
    OBJECT_FQN        VARCHAR NOT NULL,
    UPSTREAM_FQN      VARCHAR,                   -- null at the earliest layer
    DOWNSTREAM_FQN    VARCHAR,                   -- null at the terminal layer
    TRANSFORM_TYPE    VARCHAR,                   -- COPY | INSERT | MERGE | VIEW
    HAS_UPSTREAM      BOOLEAN NOT NULL DEFAULT FALSE,
    HAS_DOWNSTREAM    BOOLEAN NOT NULL DEFAULT FALSE,
    CREATED_AT        TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

COMMENT ON TABLE DTS_LINEAGE_REGISTRY IS
    'Lineage edges per object; seeded with the POSITION and TRENDED chains from edl-pnc-repo.';

-------------------------------------------------------------------------------
-- 7. DTS_CLASSIFICATION_REGISTRY -- Classification dimension (foundational).
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_CLASSIFICATION_REGISTRY (
    DATASET_ID           NUMBER NOT NULL,
    REPORT_FAMILY        VARCHAR NOT NULL,
    LAYER                VARCHAR NOT NULL,
    HAS_CLASSIFICATION   BOOLEAN NOT NULL DEFAULT FALSE,
    SENSITIVITY_LEVEL    VARCHAR,               -- PUBLIC | INTERNAL | CONFIDENTIAL | RESTRICTED
    PII_PRESENT          BOOLEAN NOT NULL DEFAULT FALSE,
    CLASSIFIED_COLUMN_PCT FLOAT  NOT NULL DEFAULT 0,  -- % of columns tagged
    CREATED_AT           TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT UQ_DTS_CLASS UNIQUE (REPORT_FAMILY, LAYER)
);

COMMENT ON TABLE DTS_CLASSIFICATION_REGISTRY IS
    'Classification coverage per object (foundational dim -- gap flag if unseeded).';

## 2 · Config tables (weights, bands, confidence factors)

In [ ]:
-------------------------------------------------------------------------------
-- ddl/02_dts_config.sql
-- v2 Data Trust Score -- scoring configuration (tunable by governance).
--
-- Three small config tables the scoring engine reads:
--   DTS_DIMENSION_WEIGHTS   : 10 dims, weights sum to 100, + flags
--   DTS_TRUST_BANDS         : 5 bands (Certified >=90 pinned; rest tunable)
--   DTS_DQ_CONFIDENCE_FACTOR: per-layer confidence multiplier on measured DQ
--
-- Values mirror app/config.py exactly (single source of truth). Seeded inline
-- here so the DDL deploy is self-contained.
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

-------------------------------------------------------------------------------
-- 1. DTS_DIMENSION_WEIGHTS
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_DIMENSION_WEIGHTS (
    DIMENSION_CODE    VARCHAR NOT NULL PRIMARY KEY,
    DIMENSION_LABEL   VARCHAR NOT NULL,
    WEIGHT            NUMBER  NOT NULL,      -- sums to 100 across rows
    IS_FOUNDATIONAL   BOOLEAN NOT NULL,      -- gap flag if unseeded/zero
    IS_DMF_MEASURED   BOOLEAN NOT NULL       -- contributes to measurable ceiling
);

INSERT INTO DTS_DIMENSION_WEIGHTS
    (DIMENSION_CODE, DIMENSION_LABEL, WEIGHT, IS_FOUNDATIONAL, IS_DMF_MEASURED)
VALUES
    ('DQ',             'Data Quality (DAMA)',      22, FALSE, TRUE),
    ('OBSERVABILITY',  'Observability',            12, FALSE, TRUE),
    ('OWNERSHIP',      'Ownership & Stewardship',  12, TRUE,  FALSE),
    ('CLASSIFICATION', 'Classification',           12, TRUE,  FALSE),
    ('AUTH_SOURCE',    'Authoritative Source',     10, TRUE,  FALSE),
    ('LINEAGE',        'Lineage',                  10, FALSE, FALSE),
    ('DEFINITIONS',    'Business Definitions',      9, FALSE, FALSE),
    ('ACTIVE_ISSUES',  'Active Issues',             5, FALSE, FALSE),
    ('USAGE',          'Usage',                     5, FALSE, FALSE),
    ('FEEDBACK',       'User Feedback',             3, FALSE, FALSE);

-- Guard: weights must sum to 100.
SELECT IFF(SUM(WEIGHT) = 100, 'OK', 'ERROR: weights sum = ' || SUM(WEIGHT)) AS weight_check
FROM DTS_DIMENSION_WEIGHTS;

-------------------------------------------------------------------------------
-- 2. DTS_TRUST_BANDS  (Certified >=90 pinned by framework; others are defaults)
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_TRUST_BANDS (
    BAND_NAME       VARCHAR NOT NULL PRIMARY KEY,
    MIN_SCORE       NUMBER  NOT NULL,   -- inclusive
    MAX_SCORE       NUMBER  NOT NULL,   -- inclusive
    SORT_ORDER      NUMBER  NOT NULL,
    HEX_COLOR       VARCHAR NOT NULL
);

INSERT INTO DTS_TRUST_BANDS (BAND_NAME, MIN_SCORE, MAX_SCORE, SORT_ORDER, HEX_COLOR)
VALUES
    ('CERTIFIED',   90, 100, 1, '#1E8E3E'),
    ('TRUSTED',     75,  89, 2, '#66BB6A'),
    ('ESTABLISHED', 60,  74, 3, '#C9A227'),
    ('PROVISIONAL', 40,  59, 4, '#ED9B0F'),
    ('AT_RISK',      0,  39, 5, '#C44545');

-------------------------------------------------------------------------------
-- 3. DTS_DQ_CONFIDENCE_FACTOR
--    Layer -> confidence tier -> multiplier applied to the measured DQ + Obs
--    dimensions. INT=Bronze, DW=Silver, PUBL=Gold (Source reserved, unused).
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_DQ_CONFIDENCE_FACTOR (
    CONFIDENCE_TIER   VARCHAR NOT NULL PRIMARY KEY,   -- SOURCE|BRONZE|SILVER|GOLD
    LAYER             VARCHAR,                         -- INT|DW|PUBL (null for SOURCE)
    CONFIDENCE_FACTOR FLOAT   NOT NULL
);

INSERT INTO DTS_DQ_CONFIDENCE_FACTOR (CONFIDENCE_TIER, LAYER, CONFIDENCE_FACTOR)
VALUES
    ('SOURCE', NULL,   1.00),
    ('BRONZE', 'INT',  0.90),
    ('SILVER', 'DW',   0.75),
    ('GOLD',   'PUBL', 0.60);

## 3 · Measured / event tables

In [ ]:
-------------------------------------------------------------------------------
-- ddl/03_dts_measured.sql
-- v2 Data Trust Score -- measured / event tables.
--
-- These capture the non-DMF measured signals feeding several dimensions:
--   DTS_DQ_RULE_RESULT      : DAMA-tagged rule outcomes (accuracy/consistency/
--                             validity seeded from IDQ; completeness/uniqueness
--                             also mirrored here from DMFs for a unified view).
--   DTS_OBSERVABILITY_INCIDENT : freshness/volume incidents (Observability +
--                             Active Issues dims).
--   DTS_USAGE_METRICS       : query/consumer counts (Usage dim).
--   DTS_USER_FEEDBACK       : consumer feedback ratings (Feedback dim).
--
-- The authoritative DMF measurements themselves live in Snowflake's DMF result
-- store and are surfaced by the bridge views in ddl/26. This file holds the
-- seeded / interim and applied-signal tables.
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

-------------------------------------------------------------------------------
-- 1. DTS_DQ_RULE_RESULT -- one row per (object, DAMA sub-dim, run).
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_DQ_RULE_RESULT (
    RESULT_ID         NUMBER IDENTITY(1,1) PRIMARY KEY,
    REPORT_FAMILY     VARCHAR NOT NULL,
    LAYER             VARCHAR NOT NULL,
    COLUMN_NAME       VARCHAR,                    -- null = table-grain rule
    DAMA_SUB_DIM      VARCHAR NOT NULL,           -- COMPLETENESS|ACCURACY|CONSISTENCY|VALIDITY|UNIQUENESS
    RULE_NAME         VARCHAR NOT NULL,
    IS_DMF_AUTOMATED  BOOLEAN NOT NULL DEFAULT FALSE,
    PASS_PCT          FLOAT,                       -- 0-100 (score contribution)
    MEASURED_VALUE    FLOAT,                       -- raw metric (e.g. null_pct)
    THRESHOLD         FLOAT,
    STATUS            VARCHAR,                      -- PASS|WARN|FAIL
    SOURCE_SYSTEM     VARCHAR DEFAULT 'IDQ',        -- IDQ|COLLIBRA|DMF
    SCORE_RUN_DATE    DATE    NOT NULL DEFAULT CURRENT_DATE(),
    CREATED_AT        TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

COMMENT ON TABLE DTS_DQ_RULE_RESULT IS
    'DAMA-tagged DQ rule outcomes per object/column/run. DMF-automated rows have IS_DMF_AUTOMATED=TRUE; others are interim IDQ/Collibra seeds.';

-------------------------------------------------------------------------------
-- 2. DTS_OBSERVABILITY_INCIDENT -- freshness/volume incidents.
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_OBSERVABILITY_INCIDENT (
    INCIDENT_ID       NUMBER IDENTITY(1,1) PRIMARY KEY,
    REPORT_FAMILY     VARCHAR NOT NULL,
    LAYER             VARCHAR NOT NULL,
    INCIDENT_TYPE     VARCHAR NOT NULL,            -- FRESHNESS|VOLUME|SCHEMA_DRIFT
    SEVERITY          VARCHAR NOT NULL,            -- P1|P2|P3
    STATUS            VARCHAR NOT NULL DEFAULT 'OPEN',  -- OPEN|RESOLVED
    DETAIL            VARCHAR,
    DETECTED_AT       TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    RESOLVED_AT       TIMESTAMP_NTZ
);

COMMENT ON TABLE DTS_OBSERVABILITY_INCIDENT IS
    'Observability incidents (freshness/volume). Open incidents reduce Observability + Active Issues scores.';

-------------------------------------------------------------------------------
-- 3. DTS_USAGE_METRICS -- consumption signals (Usage dim).
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_USAGE_METRICS (
    REPORT_FAMILY     VARCHAR NOT NULL,
    LAYER             VARCHAR NOT NULL,
    QUERY_COUNT_30D   NUMBER  NOT NULL DEFAULT 0,
    DISTINCT_USERS_30D NUMBER NOT NULL DEFAULT 0,
    LAST_QUERIED_AT   TIMESTAMP_NTZ,
    SCORE_RUN_DATE    DATE    NOT NULL DEFAULT CURRENT_DATE(),
    CONSTRAINT UQ_DTS_USAGE UNIQUE (REPORT_FAMILY, LAYER, SCORE_RUN_DATE)
);

COMMENT ON TABLE DTS_USAGE_METRICS IS
    'Per-object usage counts (30d) feeding the Usage dimension. Can later be populated from ACCOUNT_USAGE.ACCESS_HISTORY.';

-------------------------------------------------------------------------------
-- 4. DTS_USER_FEEDBACK -- consumer feedback (Feedback dim).
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_USER_FEEDBACK (
    FEEDBACK_ID       NUMBER IDENTITY(1,1) PRIMARY KEY,
    REPORT_FAMILY     VARCHAR NOT NULL,
    LAYER             VARCHAR NOT NULL,
    RATING            NUMBER,                      -- 1-5
    COMMENT_TEXT      VARCHAR,
    SUBMITTED_BY      VARCHAR,
    SUBMITTED_AT      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

COMMENT ON TABLE DTS_USER_FEEDBACK IS
    'Consumer feedback ratings feeding the Feedback dimension (lowest weight, 3).';

## 4 · Score output tables

In [ ]:
-------------------------------------------------------------------------------
-- ddl/04_dts_scores.sql
-- v2 Data Trust Score -- score output tables (written by ddl/30 scoring engine).
--
--   DTS_ELEMENT_DIMENSION_SCORE : finest grain -- (object x element x dim x run).
--                                 0-100 per cell before rollup.
--   DTS_DATASET_TRUST_SCORE     : final per-(object, run) trust score with band,
--                                 measurable ceiling, and foundational-gap flag.
--
-- Both are CREATE OR REPLACE; the scoring engine TRUNCATE+INSERTs per run_date.
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

-------------------------------------------------------------------------------
-- 1. DTS_ELEMENT_DIMENSION_SCORE
--    Element-grain dimension scores. For table-grain dimensions the
--    COLUMN_NAME is null (one row per object x dim). For DQ, rows can be per
--    column so the CDE multiplier applies at element grain.
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_ELEMENT_DIMENSION_SCORE (
    SCORE_ROW_ID          NUMBER IDENTITY(1,1) PRIMARY KEY,
    REPORT_FAMILY         VARCHAR NOT NULL,
    LAYER                 VARCHAR NOT NULL,
    COLUMN_NAME           VARCHAR,                 -- null = table-grain
    DIMENSION_CODE        VARCHAR NOT NULL,
    RAW_SCORE             FLOAT,                   -- 0-100 before confidence/CDE; NULL = not measurable
    CONFIDENCE_FACTOR     FLOAT   NOT NULL DEFAULT 1.0,   -- layer factor (DQ/Obs only)
    CRITICALITY_MULTIPLIER FLOAT  NOT NULL DEFAULT 1.0,   -- 2.0 for CDE columns
    IS_MEASURABLE         BOOLEAN NOT NULL DEFAULT TRUE,  -- FALSE = unseeded foundational
    SCORE_RUN_DATE        DATE    NOT NULL DEFAULT CURRENT_DATE(),
    CREATED_AT            TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

COMMENT ON TABLE DTS_ELEMENT_DIMENSION_SCORE IS
    'Element x dimension 0-100 scores per run, with confidence factor and CDE multiplier, before weighted rollup.';

-------------------------------------------------------------------------------
-- 2. DTS_DATASET_TRUST_SCORE -- final per-(object, run) score.
-------------------------------------------------------------------------------
CREATE OR REPLACE TABLE DTS_DATASET_TRUST_SCORE (
    REPORT_FAMILY          VARCHAR NOT NULL,
    LAYER                  VARCHAR NOT NULL,
    DATASET_FQN            VARCHAR,
    DQ_MEASUREMENT_LAYER   VARCHAR,                -- BRONZE|SILVER|GOLD
    TRUST_SCORE            FLOAT   NOT NULL,       -- 0-100 weighted, CDE-adjusted
    TRUST_BAND             VARCHAR NOT NULL,       -- from DTS_TRUST_BANDS
    MEASURABLE_CEILING     FLOAT   NOT NULL,       -- sum of weights measurable today
    MEASURED_SUBTOTAL      FLOAT   NOT NULL,       -- score from measurable dims only
    FOUNDATIONAL_GAP_FLAG  BOOLEAN NOT NULL,       -- TRUE if a foundational dim unseeded
    FOUNDATIONAL_GAPS      ARRAY,                  -- which foundational dims are missing
    SCORE_RUN_DATE         DATE    NOT NULL DEFAULT CURRENT_DATE(),
    CREATED_AT             TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT UQ_DTS_TRUST_SCORE UNIQUE (REPORT_FAMILY, LAYER, SCORE_RUN_DATE)
);

COMMENT ON TABLE DTS_DATASET_TRUST_SCORE IS
    'Final v2 trust score per (report_family, layer, run): weighted+CDE-adjusted, banded, with measurable ceiling and foundational-gap flag.';

## 5 · Clone + baseline DMF procedures

Each procedure is in its own cell (a `$$` body cannot share a cell with other statements). `SP_DTS_CLONE_SCORED_OBJECTS` is used by both tracks; `SP_DTS_ATTACH_BASELINE_DMFS` is Track A only.

In [ ]:
-------------------------------------------------------------------------------
-- ddl/20_dts_clone_and_dmf.sql
-- v2 Data Trust Score -- clone machinery + baseline DMF attachment.
--
-- Adapted from edl-pnc-repo PNC_DATA_TRUST/40_CLONE_SOURCE_SCHEMAS.sql +
-- 50_SP_ATTACH_BASELINE_DMFS.sql, but WITHOUT creating new schemas. Because
-- PNC_DEVELOPER_RL cannot CREATE SCHEMA on the shared EDLE databases, we clone
-- each scored object as an individual DTS_CLONE__<db>__<schema>__<obj> object
-- INSIDE EDLE_DW_DB.PNC_DATA:
--   - TABLES  -> zero-copy CREATE TABLE ... CLONE (cross-db clone is allowed)
--   - VIEWS   -> CTAS snapshot (views cannot be zero-copy cloned)
--
-- Production objects are never altered; DMFs attach only to the clones.
--
-- Prereqs (ACCOUNTADMIN, one-time):
--   GRANT EXECUTE DATA METRIC FUNCTION ON ACCOUNT  TO ROLE PNC_DEVELOPER_RL;
--   GRANT DATABASE ROLE SNOWFLAKE.DATA_METRIC_USER TO ROLE PNC_DEVELOPER_RL;
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

In [ ]:
session.sql("-------------------------------------------------------------------------------\n-- 1. SP_DTS_CLONE_SCORED_OBJECTS\n--    Iterates DTS_DATASET_REGISTRY (IS_IN_SCOPE), clones each object into\n--    PNC_DATA, and sets CLONE_FQN + CLONE_EXISTS. Re-runnable (CREATE OR REPLACE\n--    per clone re-snapshots current source state).\n-------------------------------------------------------------------------------\nCREATE OR REPLACE PROCEDURE SP_DTS_CLONE_SCORED_OBJECTS()\nRETURNS VARIANT\nLANGUAGE SQL\nEXECUTE AS CALLER\nAS\n$$\nDECLARE\n    cloned_count INTEGER DEFAULT 0;\n    failed_count INTEGER DEFAULT 0;\n    failures     ARRAY   DEFAULT ARRAY_CONSTRUCT();\n    src_fqn      STRING;\n    clone_nm     STRING;\n    clone_fqn    STRING;\n    is_view      BOOLEAN;\n    sql_stmt     STRING;\n    c CURSOR FOR\n        SELECT REPORT_FAMILY, LAYER, DATABASE_NAME, SCHEMA_NAME, OBJECT_NAME,\n               DATASET_FQN, IS_VIEW\n        FROM   DTS_DATASET_REGISTRY\n        WHERE  IS_IN_SCOPE = TRUE\n        ORDER BY REPORT_FAMILY, LAYER;\nBEGIN\n    FOR rec IN c DO\n        src_fqn  := rec.DATASET_FQN;\n        is_view  := rec.IS_VIEW;\n        clone_nm := 'DTS_CLONE__' || rec.DATABASE_NAME || '__'\n                                   || rec.SCHEMA_NAME  || '__'\n                                   || rec.OBJECT_NAME;\n        clone_fqn := 'EDLE_DW_DB.PNC_DATA.' || clone_nm;\n\n        BEGIN\n            IF (is_view) THEN\n                -- Views can't be zero-copy cloned: snapshot via CTAS.\n                sql_stmt := 'CREATE OR REPLACE TABLE ' || clone_fqn\n                         || ' AS SELECT * FROM ' || src_fqn;\n            ELSE\n                -- Tables: fast metadata-only zero-copy clone.\n                sql_stmt := 'CREATE OR REPLACE TABLE ' || clone_fqn\n                         || ' CLONE ' || src_fqn;\n            END IF;\n            EXECUTE IMMEDIATE :sql_stmt;\n\n            UPDATE DTS_DATASET_REGISTRY\n               SET CLONE_FQN = :clone_fqn, CLONE_EXISTS = TRUE\n             WHERE REPORT_FAMILY = rec.REPORT_FAMILY\n               AND LAYER = rec.LAYER;\n\n            cloned_count := cloned_count + 1;\n        EXCEPTION\n            WHEN OTHER THEN\n                failed_count := failed_count + 1;\n                failures := ARRAY_APPEND(failures,\n                    OBJECT_CONSTRUCT('src', :src_fqn, 'clone', :clone_fqn,\n                                     'sqlerr', SQLERRM));\n        END;\n    END FOR;\n\n    RETURN OBJECT_CONSTRUCT('cloned', :cloned_count,\n                            'failed', :failed_count,\n                            'failures', :failures);\nEND;\n$$;").collect()

In [ ]:
COMMENT ON PROCEDURE SP_DTS_CLONE_SCORED_OBJECTS() IS
    'Clone every in-scope object in DTS_DATASET_REGISTRY into PNC_DATA (CTAS for views); sets CLONE_FQN + CLONE_EXISTS.';

In [ ]:
session.sql('-------------------------------------------------------------------------------\n-- 2. SP_DTS_ATTACH_BASELINE_DMFS\n--    Attaches SNOWFLAKE.CORE.ROW_COUNT + FRESHNESS(anchor) to every clone.\n--    Uses FRESHNESS_COLUMN from the registry when set, else the ranked picker\n--    (LOADDATE #1) mirrored from edl-pnc-repo\'s SP_ATTACH_BASELINE_DMFS.\n-------------------------------------------------------------------------------\nCREATE OR REPLACE PROCEDURE SP_DTS_ATTACH_BASELINE_DMFS(\n    SCHEDULE_MINUTES INTEGER DEFAULT 1440\n)\nRETURNS VARIANT\nLANGUAGE SQL\nEXECUTE AS CALLER\nAS\n$$\nDECLARE\n    attached_count INTEGER DEFAULT 0;\n    failed_count   INTEGER DEFAULT 0;\n    failures       ARRAY   DEFAULT ARRAY_CONSTRUCT();\n    clone_fqn      STRING;\n    clone_nm       STRING;\n    date_col       STRING;\n    sql_stmt       STRING;\n    c CURSOR FOR\n        SELECT CLONE_FQN, OBJECT_NAME, DATABASE_NAME, SCHEMA_NAME, FRESHNESS_COLUMN\n        FROM   DTS_DATASET_REGISTRY\n        WHERE  IS_IN_SCOPE = TRUE AND CLONE_EXISTS = TRUE\n        ORDER BY REPORT_FAMILY, LAYER;\nBEGIN\n    FOR rec IN c DO\n        clone_fqn := rec.CLONE_FQN;\n        clone_nm  := SPLIT_PART(rec.CLONE_FQN, \'.\', 3);\n        date_col  := rec.FRESHNESS_COLUMN;\n\n        BEGIN\n            -- Schedule must be set before the first ADD DATA METRIC FUNCTION.\n            sql_stmt := \'ALTER TABLE \' || clone_fqn\n                     || \' SET DATA_METRIC_SCHEDULE = \'\'\' || :SCHEDULE_MINUTES || \' MINUTE\'\'\';\n            EXECUTE IMMEDIATE :sql_stmt;\n\n            -- Table-level ROW_COUNT (guards against truncate/empty loads).\n            sql_stmt := \'ALTER TABLE \' || clone_fqn\n                     || \' ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.ROW_COUNT ON ()\';\n            EXECUTE IMMEDIATE :sql_stmt;\n\n            -- Freshness anchor: registry column if present, else ranked picker\n            -- against the clone\'s own INFORMATION_SCHEMA (clone lives in PNC_DATA).\n            IF (date_col IS NULL) THEN\n                EXECUTE IMMEDIATE\n                    \'SELECT COLUMN_NAME FROM EDLE_DW_DB.INFORMATION_SCHEMA.COLUMNS \'\n                    || \'WHERE TABLE_SCHEMA = \'\'PNC_DATA\'\' AND TABLE_NAME = \'\'\' || clone_nm || \'\'\' \'\n                    || \'AND DATA_TYPE IN (\'\'DATE\'\',\'\'TIMESTAMP_NTZ\'\',\'\'TIMESTAMP_LTZ\'\',\'\'TIMESTAMP_TZ\'\') \'\n                    || \'ORDER BY CASE \'\n                    || \'  WHEN COLUMN_NAME ILIKE \'\'LOADDATE\'\'     THEN 1 \'\n                    || \'  WHEN COLUMN_NAME ILIKE \'\'LOAD\\\\_DATE\'\'  THEN 1 \'\n                    || \'  WHEN COLUMN_NAME ILIKE \'\'LOAD\\\\_TS\'\'    THEN 1 \'\n                    || \'  WHEN COLUMN_NAME ILIKE \'\'LAST\\\\_LOADED%\'\' THEN 2 \'\n                    || \'  WHEN COLUMN_NAME ILIKE \'\'INGEST%\'\'      THEN 2 \'\n                    || \'  WHEN COLUMN_NAME ILIKE \'\'UPDATED\\\\_AT\'\' THEN 3 \'\n                    || \'  WHEN COLUMN_NAME ILIKE \'\'CREATED\\\\_AT\'\' THEN 4 \'\n                    || \'  WHEN COLUMN_NAME ILIKE \'\'REPORT\\\\_RUN\\\\_DATE\'\' THEN 5 \'\n                    || \'  WHEN COLUMN_NAME ILIKE \'\'EFFECTIVEDATE\'\' THEN 6 \'\n                    || \'  ELSE 99 END, ORDINAL_POSITION LIMIT 1\';\n                SELECT $1 INTO :date_col FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));\n            END IF;\n\n            IF (date_col IS NOT NULL) THEN\n                sql_stmt := \'ALTER TABLE \' || clone_fqn\n                         || \' ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.FRESHNESS ON ("\'\n                         || date_col || \'")\';\n                EXECUTE IMMEDIATE :sql_stmt;\n            END IF;\n\n            attached_count := attached_count + 1;\n        EXCEPTION\n            WHEN OTHER THEN\n                failed_count := failed_count + 1;\n                failures := ARRAY_APPEND(failures,\n                    OBJECT_CONSTRUCT(\'clone\', :clone_fqn, \'sqlerr\', SQLERRM));\n        END;\n    END FOR;\n\n    RETURN OBJECT_CONSTRUCT(\'attached\', :attached_count,\n                            \'failed\', :failed_count,\n                            \'schedule_minutes\', :SCHEDULE_MINUTES,\n                            \'failures\', :failures);\nEND;\n$$;').collect()

In [ ]:
COMMENT ON PROCEDURE SP_DTS_ATTACH_BASELINE_DMFS(INTEGER) IS
    'Attach ROW_COUNT + FRESHNESS to every DTS_CLONE__* in DTS_DATASET_REGISTRY (CLONE_EXISTS=TRUE).';

## 6 · Key-column DMF procedure  _(Track A only)_

In [ ]:
-------------------------------------------------------------------------------
-- ddl/21_dts_key_dmfs.sql
-- v2 Data Trust Score -- key-column DMFs (completeness + uniqueness).
--
-- On top of the ROW_COUNT + FRESHNESS baseline (ddl/20), this attaches the
-- DAMA-relevant column DMFs to each clone, driven by KEY_COLUMNS in
-- DTS_DATASET_REGISTRY:
--   - SNOWFLAKE.CORE.NULL_COUNT      ON (<each key column>)      -> Completeness
--   - SNOWFLAKE.CORE.DUPLICATE_COUNT ON (<composite key tuple>)  -> Uniqueness
--
-- ADD DATA METRIC FUNCTION is idempotent-unsafe (throws on duplicate), so the
-- per-DMF ADD is wrapped in its own block and duplicate errors are tolerated.
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

In [ ]:
session.sql('CREATE OR REPLACE PROCEDURE SP_DTS_ATTACH_KEY_DMFS()\nRETURNS VARIANT\nLANGUAGE SQL\nEXECUTE AS CALLER\nAS\n$$\nDECLARE\n    attached_count INTEGER DEFAULT 0;\n    failed_count   INTEGER DEFAULT 0;\n    failures       ARRAY   DEFAULT ARRAY_CONSTRUCT();\n    clone_fqn      STRING;\n    key_cols       ARRAY;\n    n_keys         INTEGER;\n    i              INTEGER;\n    one_col        STRING;\n    tuple_cols     STRING;\n    sql_stmt       STRING;\n    c CURSOR FOR\n        SELECT CLONE_FQN, KEY_COLUMNS\n        FROM   DTS_DATASET_REGISTRY\n        WHERE  IS_IN_SCOPE = TRUE AND CLONE_EXISTS = TRUE\n          AND  KEY_COLUMNS IS NOT NULL\n        ORDER BY REPORT_FAMILY, LAYER;\nBEGIN\n    FOR rec IN c DO\n        clone_fqn := rec.CLONE_FQN;\n        key_cols  := rec.KEY_COLUMNS;\n        n_keys    := ARRAY_SIZE(key_cols);\n\n        -- NULL_COUNT per individual key column (completeness).\n        i := 0;\n        WHILE (i < n_keys) DO\n            one_col := GET(key_cols, i)::STRING;\n            BEGIN\n                sql_stmt := \'ALTER TABLE \' || clone_fqn\n                         || \' ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.NULL_COUNT ON ("\'\n                         || one_col || \'")\';\n                EXECUTE IMMEDIATE :sql_stmt;\n                attached_count := attached_count + 1;\n            EXCEPTION\n                WHEN OTHER THEN\n                    failed_count := failed_count + 1;\n                    failures := ARRAY_APPEND(failures,\n                        OBJECT_CONSTRUCT(\'clone\', :clone_fqn, \'dmf\', \'NULL_COUNT(\'||:one_col||\')\',\n                                         \'sqlerr\', SQLERRM));\n            END;\n            i := i + 1;\n        END WHILE;\n\n        -- DUPLICATE_COUNT on the composite key tuple (uniqueness). Columns are\n        -- double-quoted so spaced PUBL-view names (e.g. "Position ID") resolve.\n        tuple_cols := \'"\' || ARRAY_TO_STRING(key_cols, \'", "\') || \'"\';\n        BEGIN\n            sql_stmt := \'ALTER TABLE \' || clone_fqn\n                     || \' ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.DUPLICATE_COUNT ON (\'\n                     || tuple_cols || \')\';\n            EXECUTE IMMEDIATE :sql_stmt;\n            attached_count := attached_count + 1;\n        EXCEPTION\n            WHEN OTHER THEN\n                failed_count := failed_count + 1;\n                failures := ARRAY_APPEND(failures,\n                    OBJECT_CONSTRUCT(\'clone\', :clone_fqn, \'dmf\', \'DUPLICATE_COUNT(\'||:tuple_cols||\')\',\n                                     \'sqlerr\', SQLERRM));\n        END;\n    END FOR;\n\n    RETURN OBJECT_CONSTRUCT(\'attached\', :attached_count,\n                            \'failed\', :failed_count,\n                            \'failures\', :failures);\nEND;\n$$;').collect()

In [ ]:
COMMENT ON PROCEDURE SP_DTS_ATTACH_KEY_DMFS() IS
    'Attach NULL_COUNT (per key col) + DUPLICATE_COUNT (composite key) to each clone, from DTS_DATASET_REGISTRY.KEY_COLUMNS.';

## 7 · Measurement store + populators  _(must run before step 8)_

`DTS_DMF_MEASUREMENTS` + Track B `SP_DTS_COMPUTE_DQ_MEASUREMENTS` (no grant) and Track A `SP_DTS_SYNC_NATIVE_DMF_RESULTS`.

In [ ]:
-------------------------------------------------------------------------------
-- ddl/25_dts_dmf_measurements.sql
-- v2 Data Trust Score -- unified DMF measurement store + two populators.
--
-- WHY: PNC_DEVELOPER_RL does not (yet) have EXECUTE DATA METRIC FUNCTION on the
-- account, so native DMFs can't attach. To stay switchable, both tracks write
-- to ONE normalized table (DTS_DMF_MEASUREMENTS) and the bridge view in ddl/26
-- reads only that table -- so nothing references SNOWFLAKE.LOCAL directly and
-- there is no privilege error in manual mode.
--
--   Track B (default, no grant needed):  SP_DTS_COMPUTE_DQ_MEASUREMENTS()
--       computes ROW_COUNT / FRESHNESS / NULL_COUNT(keys) / DUPLICATE_COUNT in
--       plain SQL over the DTS_CLONE__* tables (which PNC_DEVELOPER_RL owns).
--
--   Track A (once granted):              SP_DTS_SYNC_NATIVE_DMF_RESULTS()
--       copies the latest native DMF readings from
--       SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS into the same table.
--       Requires the ddl/20 + 21 ALTER ... ADD DATA METRIC FUNCTION to have run
--       (needs EXECUTE DATA METRIC FUNCTION + SNOWFLAKE.DATA_METRIC_USER).
--
-- Switch tracks by choosing which populator you schedule; the bridge view and
-- scoring engine don't change. Metric names mirror SNOWFLAKE.CORE.* so the
-- bridge logic is identical for both tracks.
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

-------------------------------------------------------------------------------
-- 1. Unified measurement store (both tracks INSERT here).
-------------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS DTS_DMF_MEASUREMENTS (
    CLONE_FQN         VARCHAR       NOT NULL,
    REPORT_FAMILY     VARCHAR       NOT NULL,
    LAYER             VARCHAR       NOT NULL,
    METRIC_NAME       VARCHAR       NOT NULL,   -- SNOWFLAKE.CORE.ROW_COUNT etc.
    COLUMN_NAME       VARCHAR,                   -- NULL for table-grain metrics
    VALUE             FLOAT,
    MEASUREMENT_TIME  TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    SOURCE_TRACK      VARCHAR       NOT NULL     -- MANUAL | NATIVE
);

COMMENT ON TABLE DTS_DMF_MEASUREMENTS IS
    'Unified DMF measurement store. Track B (SP_DTS_COMPUTE_DQ_MEASUREMENTS) or Track A (SP_DTS_SYNC_NATIVE_DMF_RESULTS) populate it; the bridge view reads only this table.';

In [ ]:
session.sql('-------------------------------------------------------------------------------\n-- 2. TRACK B -- manual SQL computation over the clones (no account grant).\n--    Metric names mirror the native DMFs so ddl/26 treats both identically.\n-------------------------------------------------------------------------------\nCREATE OR REPLACE PROCEDURE SP_DTS_COMPUTE_DQ_MEASUREMENTS()\nRETURNS VARIANT\nLANGUAGE SQL\nEXECUTE AS CALLER\nAS\n$$\nDECLARE\n    rows_written INTEGER DEFAULT 0;\n    failed_count INTEGER DEFAULT 0;\n    failures     ARRAY   DEFAULT ARRAY_CONSTRUCT();\n    clone_fqn    STRING;\n    fam          STRING;\n    lyr          STRING;\n    fcol         STRING;\n    key_cols     ARRAY;\n    n_keys       INTEGER;\n    i            INTEGER;\n    one_col      STRING;\n    tuple_cols   STRING;\n    tuple_cols_q STRING;\n    sql_stmt     STRING;\n    c CURSOR FOR\n        SELECT CLONE_FQN, REPORT_FAMILY, LAYER, FRESHNESS_COLUMN, KEY_COLUMNS\n        FROM   DTS_DATASET_REGISTRY\n        WHERE  IS_IN_SCOPE = TRUE AND CLONE_EXISTS = TRUE\n        ORDER BY REPORT_FAMILY, LAYER;\nBEGIN\n    -- Fresh snapshot for this run: clear prior MANUAL rows.\n    DELETE FROM DTS_DMF_MEASUREMENTS WHERE SOURCE_TRACK = \'MANUAL\';\n\n    FOR rec IN c DO\n        clone_fqn := rec.CLONE_FQN;\n        fam := rec.REPORT_FAMILY; lyr := rec.LAYER;\n        fcol := rec.FRESHNESS_COLUMN; key_cols := rec.KEY_COLUMNS;\n\n        BEGIN\n            -- ROW_COUNT (table-grain)\n            sql_stmt := \'INSERT INTO DTS_DMF_MEASUREMENTS \'\n                || \'(CLONE_FQN,REPORT_FAMILY,LAYER,METRIC_NAME,COLUMN_NAME,VALUE,SOURCE_TRACK) \'\n                || \'SELECT \'\'\' || clone_fqn || \'\'\',\'\'\' || fam || \'\'\',\'\'\' || lyr || \'\'\',\'\n                || \'\'\'SNOWFLAKE.CORE.ROW_COUNT\'\',NULL,(SELECT COUNT(*) FROM \' || clone_fqn || \'),\'\'MANUAL\'\'\';\n            EXECUTE IMMEDIATE :sql_stmt;\n            rows_written := rows_written + 1;\n\n            -- NULL_COUNT per key column (completeness)\n            n_keys := ARRAY_SIZE(key_cols);\n            i := 0;\n            WHILE (i < n_keys) DO\n                one_col := GET(key_cols, i)::STRING;\n                sql_stmt := \'INSERT INTO DTS_DMF_MEASUREMENTS \'\n                    || \'(CLONE_FQN,REPORT_FAMILY,LAYER,METRIC_NAME,COLUMN_NAME,VALUE,SOURCE_TRACK) \'\n                    || \'SELECT \'\'\' || clone_fqn || \'\'\',\'\'\' || fam || \'\'\',\'\'\' || lyr || \'\'\',\'\n                    || \'\'\'SNOWFLAKE.CORE.NULL_COUNT\'\',\'\'\' || one_col || \'\'\',\'\n                    || \'(SELECT COUNT(*) FROM \' || clone_fqn || \' WHERE "\' || one_col || \'" IS NULL),\'\'MANUAL\'\'\';\n                EXECUTE IMMEDIATE :sql_stmt;\n                rows_written := rows_written + 1;\n                i := i + 1;\n            END WHILE;\n\n            -- DUPLICATE_COUNT on the composite key (uniqueness): extra rows beyond\n            -- the first in each key group. Guarded: skip when there are no keys\n            -- (empty GROUP BY would be a syntax error).\n            IF (n_keys > 0) THEN\n                tuple_cols   := ARRAY_TO_STRING(key_cols, \', \');\n                tuple_cols_q := \'"\' || ARRAY_TO_STRING(key_cols, \'", "\') || \'"\';\n                sql_stmt := \'INSERT INTO DTS_DMF_MEASUREMENTS \'\n                    || \'(CLONE_FQN,REPORT_FAMILY,LAYER,METRIC_NAME,COLUMN_NAME,VALUE,SOURCE_TRACK) \'\n                    || \'SELECT \'\'\' || clone_fqn || \'\'\',\'\'\' || fam || \'\'\',\'\'\' || lyr || \'\'\',\'\n                    || \'\'\'SNOWFLAKE.CORE.DUPLICATE_COUNT\'\',\'\'\' || tuple_cols || \'\'\',\'\n                    || \'COALESCE((SELECT SUM(c-1) FROM (SELECT COUNT(*) c FROM \' || clone_fqn\n                    || \' GROUP BY \' || tuple_cols_q || \' HAVING COUNT(*) > 1)),0),\'\'MANUAL\'\'\';\n                EXECUTE IMMEDIATE :sql_stmt;\n                rows_written := rows_written + 1;\n            END IF;\n\n        EXCEPTION\n            WHEN OTHER THEN\n                failed_count := failed_count + 1;\n                failures := ARRAY_APPEND(failures,\n                    OBJECT_CONSTRUCT(\'clone\', :clone_fqn, \'stage\', \'core_metrics\', \'sqlerr\', SQLERRM));\n        END;\n\n        -- FRESHNESS (seconds since latest anchor) -- isolated so a missing anchor\n        -- column (e.g. a PUBL view without LOADDATE) doesn\'t drop the core metrics.\n        IF (fcol IS NOT NULL) THEN\n            BEGIN\n                sql_stmt := \'INSERT INTO DTS_DMF_MEASUREMENTS \'\n                    || \'(CLONE_FQN,REPORT_FAMILY,LAYER,METRIC_NAME,COLUMN_NAME,VALUE,SOURCE_TRACK) \'\n                    || \'SELECT \'\'\' || clone_fqn || \'\'\',\'\'\' || fam || \'\'\',\'\'\' || lyr || \'\'\',\'\n                    || \'\'\'SNOWFLAKE.CORE.FRESHNESS\'\',\'\'\' || fcol || \'\'\',\'\n                    || \'DATEDIFF(\'\'second\'\', MAX("\' || fcol || \'"), CURRENT_TIMESTAMP()),\'\'MANUAL\'\' FROM \' || clone_fqn\n                    || \' HAVING MAX("\' || fcol || \'") IS NOT NULL\';\n                EXECUTE IMMEDIATE :sql_stmt;\n                rows_written := rows_written + 1;\n            EXCEPTION\n                WHEN OTHER THEN\n                    failed_count := failed_count + 1;\n                    failures := ARRAY_APPEND(failures,\n                        OBJECT_CONSTRUCT(\'clone\', :clone_fqn, \'stage\', \'freshness\',\n                                         \'col\', :fcol, \'sqlerr\', SQLERRM));\n            END;\n        END IF;\n    END FOR;\n\n    RETURN OBJECT_CONSTRUCT(\'track\',\'MANUAL\',\'rows_written\',:rows_written,\n                            \'failed\',:failed_count,\'failures\',:failures);\nEND;\n$$;').collect()

In [ ]:
COMMENT ON PROCEDURE SP_DTS_COMPUTE_DQ_MEASUREMENTS() IS
    'Track B: compute ROW_COUNT/FRESHNESS/NULL_COUNT(keys)/DUPLICATE_COUNT in plain SQL over the clones; write MANUAL rows to DTS_DMF_MEASUREMENTS. No account grant needed.';

In [ ]:
session.sql("-------------------------------------------------------------------------------\n-- 3. TRACK A -- sync native DMF results (run only once the grants exist and\n--    ddl/20 + 21 have attached DMFs to the clones).\n-------------------------------------------------------------------------------\nCREATE OR REPLACE PROCEDURE SP_DTS_SYNC_NATIVE_DMF_RESULTS()\nRETURNS VARIANT\nLANGUAGE SQL\nEXECUTE AS CALLER\nAS\n$$\nDECLARE\n    rows_written INTEGER DEFAULT 0;\nBEGIN\n    DELETE FROM DTS_DMF_MEASUREMENTS WHERE SOURCE_TRACK = 'NATIVE';\n\n    INSERT INTO DTS_DMF_MEASUREMENTS\n        (CLONE_FQN, REPORT_FAMILY, LAYER, METRIC_NAME, COLUMN_NAME, VALUE, MEASUREMENT_TIME, SOURCE_TRACK)\n    SELECT\n        r.CLONE_FQN, d.REPORT_FAMILY, d.LAYER,\n        UPPER(r.METRIC_DATABASE || '.' || r.METRIC_SCHEMA || '.' || r.METRIC_NAME),\n        ARRAY_TO_STRING(r.ARGUMENT_NAMES, ','),\n        r.VALUE, r.MEASUREMENT_TIME, 'NATIVE'\n    FROM (\n        SELECT TABLE_DATABASE || '.' || TABLE_SCHEMA || '.' || TABLE_NAME AS CLONE_FQN,\n               METRIC_DATABASE, METRIC_SCHEMA, METRIC_NAME, ARGUMENT_NAMES, VALUE, MEASUREMENT_TIME,\n               ROW_NUMBER() OVER (\n                 PARTITION BY TABLE_DATABASE, TABLE_SCHEMA, TABLE_NAME, METRIC_NAME,\n                              ARRAY_TO_STRING(ARGUMENT_NAMES, ',')\n                 ORDER BY MEASUREMENT_TIME DESC) AS rn\n        FROM SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS\n    ) r\n    JOIN DTS_DATASET_REGISTRY d ON d.CLONE_FQN = r.CLONE_FQN\n    WHERE r.rn = 1 AND d.IS_IN_SCOPE = TRUE;\n\n    rows_written := SQLROWCOUNT;\n    RETURN OBJECT_CONSTRUCT('track','NATIVE','rows_written',:rows_written);\nEND;\n$$;").collect()

In [ ]:
COMMENT ON PROCEDURE SP_DTS_SYNC_NATIVE_DMF_RESULTS() IS
    'Track A: copy latest native DMF readings from SNOWFLAKE.LOCAL into DTS_DMF_MEASUREMENTS. Needs EXECUTE DATA METRIC FUNCTION + SNOWFLAKE.DATA_METRIC_USER and ddl/20+21 attached.';

## 8 · Bridge views  _(reads DTS_DMF_MEASUREMENTS from step 7)_

In [ ]:
-------------------------------------------------------------------------------
-- ddl/26_dts_bridge_views.sql
-- v2 Data Trust Score -- DMF bridge views (raw DMF output -> 0-100 dim scores).
--
-- MUST be deployed AFTER ddl/25 (it selects from DTS_DMF_MEASUREMENTS, which
-- ddl/25 creates). Both DMF tracks write to that unified table:
--   Track B -- SP_DTS_COMPUTE_DQ_MEASUREMENTS (plain SQL, no account grant)
--   Track A -- SP_DTS_SYNC_NATIVE_DMF_RESULTS (copies SNOWFLAKE.LOCAL results)
-- This view reads ONLY DTS_DMF_MEASUREMENTS, so it never references
-- SNOWFLAKE.LOCAL directly and works regardless of DMF privileges. Switch
-- tracks by choosing which populator SP you run.
-- These two views:
--   1. DTS_VW_DMF_LATEST_MEASUREMENTS -- latest reading per (report_family,
--      layer, metric, column) from DTS_DMF_MEASUREMENTS, with inline PASS/FAIL.
--   2. DTS_VW_DMF_DIMENSION_SCORES    -- per (report_family, layer) 0-100 scores
--      for the DMF-measured pieces of two dimensions:
--        DQ (DAMA): completeness (NULL_COUNT), uniqueness (DUPLICATE_COUNT)
--        OBSERVABILITY: freshness (FRESHNESS vs SLA), volume (ROW_COUNT >= 1)
--      plus an ACTIVE_ISSUES contribution from DMF FAIL counts.
--
-- The scoring engine (ddl/30) reads DTS_VW_DMF_DIMENSION_SCORES and blends it
-- with the registry-driven dimensions, applying the per-layer confidence factor.
--
-- SLA per family (freshness): POSITION weekly=168h, TRENDED monthly=840h.
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

-------------------------------------------------------------------------------
-- 1. DTS_VW_DMF_LATEST_MEASUREMENTS
-------------------------------------------------------------------------------
CREATE OR REPLACE VIEW DTS_VW_DMF_LATEST_MEASUREMENTS AS
WITH source AS (
    SELECT
        m.CLONE_FQN
      , m.METRIC_NAME
      , NULLIF(m.COLUMN_NAME, '')            AS COLUMN_NAME
      , m.VALUE
      , m.MEASUREMENT_TIME
      , m.SOURCE_TRACK
      , d.REPORT_FAMILY
      , d.LAYER
      , d.DATASET_FQN
      , d.DQ_MEASUREMENT_LAYER
    FROM DTS_DMF_MEASUREMENTS m
    JOIN DTS_DATASET_REGISTRY d
      ON d.CLONE_FQN = m.CLONE_FQN
    WHERE d.IS_IN_SCOPE = TRUE
)
, ranked AS (
    SELECT s.*,
           ROW_NUMBER() OVER (
             PARTITION BY REPORT_FAMILY, LAYER, METRIC_NAME, COALESCE(COLUMN_NAME, '<table>')
             ORDER BY MEASUREMENT_TIME DESC
           ) AS rn
    FROM source s
)
SELECT
    r.CLONE_FQN
  , r.REPORT_FAMILY
  , r.LAYER
  , r.DATASET_FQN                          -- original source FQN
  , r.DQ_MEASUREMENT_LAYER
  , r.METRIC_NAME
  , r.COLUMN_NAME
  , r.VALUE
  , r.MEASUREMENT_TIME
  , r.SOURCE_TRACK
  -- SLA (seconds) for freshness, by family.
  , CASE r.REPORT_FAMILY
        WHEN 'POSITION_REPORT' THEN 168 * 3600
        WHEN 'TRENDED_REPORT'  THEN 840 * 3600
        ELSE 24 * 3600
    END                                                                       AS SLA_SECS
  , CASE
        WHEN r.METRIC_NAME = 'SNOWFLAKE.CORE.NULL_COUNT'      AND r.VALUE <= 0 THEN 'PASS'
        WHEN r.METRIC_NAME = 'SNOWFLAKE.CORE.DUPLICATE_COUNT' AND r.VALUE <= 0 THEN 'PASS'
        WHEN r.METRIC_NAME = 'SNOWFLAKE.CORE.ROW_COUNT'       AND r.VALUE >= 1 THEN 'PASS'
        WHEN r.METRIC_NAME = 'SNOWFLAKE.CORE.FRESHNESS'
             AND r.VALUE <= (CASE r.REPORT_FAMILY
                                WHEN 'POSITION_REPORT' THEN 168 * 3600
                                WHEN 'TRENDED_REPORT'  THEN 840 * 3600
                                ELSE 24 * 3600 END)                           THEN 'PASS'
        ELSE 'FAIL'
    END                                                                       AS STATUS
FROM ranked r
WHERE r.rn = 1;

COMMENT ON VIEW DTS_VW_DMF_LATEST_MEASUREMENTS IS
    'Latest DMF reading per (clone, metric, column) joined to DTS_DATASET_REGISTRY; source FQN recovered, PASS/FAIL inline.';

-------------------------------------------------------------------------------
-- 2. DTS_VW_DMF_DIMENSION_SCORES
--    Per (report_family, layer): DMF-derived 0-100 for DQ sub-dims +
--    Observability sub-dims + an Active Issues contribution.
-------------------------------------------------------------------------------
CREATE OR REPLACE VIEW DTS_VW_DMF_DIMENSION_SCORES AS
WITH m AS (
    SELECT * FROM DTS_VW_DMF_LATEST_MEASUREMENTS
)
, agg AS (
    SELECT
        REPORT_FAMILY
      , LAYER
      , DQ_MEASUREMENT_LAYER
      , MAX(IFF(METRIC_NAME = 'SNOWFLAKE.CORE.ROW_COUNT', VALUE, NULL))        AS ROW_COUNT_V
      , MAX(IFF(METRIC_NAME = 'SNOWFLAKE.CORE.FRESHNESS', VALUE, NULL))        AS FRESHNESS_SECS
      , MAX(SLA_SECS)                                                          AS SLA_SECS
      -- completeness: worst NULL_COUNT across key columns vs row count
      , MAX(IFF(METRIC_NAME = 'SNOWFLAKE.CORE.NULL_COUNT', VALUE, NULL))       AS MAX_NULLS
      -- uniqueness: worst DUPLICATE_COUNT across composite key
      , MAX(IFF(METRIC_NAME = 'SNOWFLAKE.CORE.DUPLICATE_COUNT', VALUE, NULL))  AS MAX_DUPS
      , COUNT(*)                                                               AS CHECKS_TOTAL
      , SUM(IFF(STATUS = 'FAIL', 1, 0))                                        AS FAIL_COUNT
    FROM m
    GROUP BY REPORT_FAMILY, LAYER, DQ_MEASUREMENT_LAYER
)
SELECT
    REPORT_FAMILY
  , LAYER
  , DQ_MEASUREMENT_LAYER

  -- DAMA completeness: 100 when no nulls; else decay by null fraction.
  , CASE
        WHEN MAX_NULLS IS NULL THEN NULL
        WHEN ROW_COUNT_V IS NULL OR ROW_COUNT_V = 0 THEN IFF(MAX_NULLS = 0, 100, 50)
        ELSE GREATEST(0, ROUND(100 - 100.0 * MAX_NULLS / ROW_COUNT_V, 1))
    END                                                                       AS DQ_COMPLETENESS_SCORE

  -- DAMA uniqueness: 100 when no dups on the composite key; else 0.
  , CASE
        WHEN MAX_DUPS IS NULL THEN NULL
        ELSE IFF(MAX_DUPS <= 0, 100, 0)
    END                                                                       AS DQ_UNIQUENESS_SCORE

  -- Observability freshness: 100 within SLA, linear decay to 0 at 3x SLA.
  , CASE
        WHEN FRESHNESS_SECS IS NULL THEN NULL
        WHEN FRESHNESS_SECS <= SLA_SECS THEN 100
        WHEN FRESHNESS_SECS >= 3 * SLA_SECS THEN 0
        ELSE ROUND(100 - 100.0 * (FRESHNESS_SECS - SLA_SECS) / (2.0 * SLA_SECS), 1)
    END                                                                       AS OBS_FRESHNESS_SCORE

  -- Observability volume: 100 if the table has rows, else 0.
  , CASE
        WHEN ROW_COUNT_V IS NULL THEN NULL
        ELSE IFF(ROW_COUNT_V >= 1, 100, 0)
    END                                                                       AS OBS_VOLUME_SCORE

  -- Active Issues contribution from DMF fails (P3-equivalent, -25 each).
  , GREATEST(0, 100 - 25 * COALESCE(FAIL_COUNT, 0))                           AS ACTIVE_ISSUES_DMF_SCORE

  , ROW_COUNT_V
  , FRESHNESS_SECS
  , SLA_SECS
  , MAX_NULLS
  , MAX_DUPS
  , CHECKS_TOTAL
  , FAIL_COUNT
  , CURRENT_TIMESTAMP()                                                       AS COMPUTED_AT
FROM agg;

COMMENT ON VIEW DTS_VW_DMF_DIMENSION_SCORES IS
    'Per (report_family, layer) DMF-derived 0-100 sub-scores: DQ completeness/uniqueness, Observability freshness/volume, Active Issues.';

## 9 · Scoring engine (`SP_DTS_COMPUTE_SCORES` + latest view)

In [ ]:
-------------------------------------------------------------------------------
-- ddl/30_dts_scoring_engine.sql
-- v2 Data Trust Score -- scoring engine.
--
-- SP_DTS_COMPUTE_SCORES(run_date) computes, per scored object
-- (report_family x layer):
--   1. Element-grain DQ completeness for each key column, CDE-weighted (2x for
--      IS_CDE columns) -> a CDE-weighted completeness sub-score.
--   2. DQ dimension = average of available DAMA sub-dims:
--        completeness (CDE-weighted, DMF), uniqueness (DMF),
--        accuracy/consistency/validity (seeded from DTS_DQ_RULE_RESULT).
--   3. Observability dimension = avg(freshness, volume) with open-incident
--      penalties.
--   4. DQ + Observability RAW scores are multiplied by the per-layer confidence
--      factor (INT/Bronze 0.90, DW/Silver 0.75, PUBL/Gold 0.60).
--   5. Registry-driven dims (Ownership, Classification, Auth Source, Lineage,
--      Definitions, Active Issues, Usage, Feedback) scored from their tables.
--   6. Weighted rollup over the 10 dims (blanks = 0) -> TRUST_SCORE (0-100).
--   7. MEASURABLE_CEILING = sum of weights of dims with a non-null score.
--   8. FOUNDATIONAL_GAP_FLAG = TRUE if any foundational dim
--      (Ownership/Classification/Auth Source) is unseeded/zero.
--   9. TRUST_BAND from DTS_TRUST_BANDS.
--
-- Writes DTS_ELEMENT_DIMENSION_SCORE (audit grain) + DTS_DATASET_TRUST_SCORE
-- (final). Idempotent per run_date (deletes that day's rows first).
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

In [ ]:
session.sql("CREATE OR REPLACE PROCEDURE SP_DTS_COMPUTE_SCORES(RUN_DATE DATE DEFAULT CURRENT_DATE())\nRETURNS VARIANT\nLANGUAGE SQL\nEXECUTE AS CALLER\nAS\n$$\nDECLARE\n    n_dim_rows   INTEGER DEFAULT 0;\n    n_score_rows INTEGER DEFAULT 0;\nBEGIN\n    -- Idempotency: clear this run_date.\n    DELETE FROM DTS_ELEMENT_DIMENSION_SCORE WHERE SCORE_RUN_DATE = :RUN_DATE;\n    DELETE FROM DTS_DATASET_TRUST_SCORE     WHERE SCORE_RUN_DATE = :RUN_DATE;\n\n    ---------------------------------------------------------------------------\n    -- Build the per-(family, layer, dimension) score set, then materialize it\n    -- into DTS_ELEMENT_DIMENSION_SCORE (table grain; COLUMN_NAME = NULL).\n    -- CONFIDENCE_FACTOR is the layer factor for DQ/OBSERVABILITY, 1.0 otherwise.\n    -- RAW_SCORE is pre-confidence, 0-100 (or NULL when not measurable).\n    ---------------------------------------------------------------------------\n    INSERT INTO DTS_ELEMENT_DIMENSION_SCORE\n        (REPORT_FAMILY, LAYER, COLUMN_NAME, DIMENSION_CODE, RAW_SCORE,\n         CONFIDENCE_FACTOR, CRITICALITY_MULTIPLIER, IS_MEASURABLE, SCORE_RUN_DATE)\n    WITH reg AS (\n        SELECT REPORT_FAMILY, LAYER, DQ_MEASUREMENT_LAYER, CLONE_FQN\n        FROM   DTS_DATASET_REGISTRY\n        WHERE  IS_IN_SCOPE = TRUE\n    )\n    , cf AS (   -- confidence factor per layer\n        SELECT LAYER, CONFIDENCE_FACTOR FROM DTS_DQ_CONFIDENCE_FACTOR WHERE LAYER IS NOT NULL\n    )\n    , dmf AS (  -- DMF-derived sub-scores per object\n        SELECT * FROM DTS_VW_DMF_DIMENSION_SCORES\n    )\n    ---------------------------------------------------------------------------\n    -- DQ: element-grain CDE-weighted completeness across key columns.\n    ---------------------------------------------------------------------------\n    , null_by_col AS (\n        SELECT\n            lm.REPORT_FAMILY, lm.LAYER, lm.COLUMN_NAME,\n            lm.VALUE AS NULL_CNT\n        FROM DTS_VW_DMF_LATEST_MEASUREMENTS lm\n        WHERE lm.METRIC_NAME = 'SNOWFLAKE.CORE.NULL_COUNT'\n    )\n    , rowcount_by_obj AS (\n        SELECT REPORT_FAMILY, LAYER, ROW_COUNT_V\n        FROM dmf\n    )\n    , comp_col AS (\n        SELECT\n            n.REPORT_FAMILY, n.LAYER, n.COLUMN_NAME,\n            CASE\n                WHEN rc.ROW_COUNT_V IS NULL OR rc.ROW_COUNT_V = 0\n                     THEN IFF(n.NULL_CNT = 0, 100, 50)\n                ELSE GREATEST(0, 100 - 100.0 * n.NULL_CNT / rc.ROW_COUNT_V)\n            END AS COMPLETENESS_SCORE,\n            COALESCE(e.CRITICALITY_MULTIPLIER, 1.0) AS CDE_MULT\n        FROM null_by_col n\n        LEFT JOIN rowcount_by_obj rc\n               ON rc.REPORT_FAMILY = n.REPORT_FAMILY AND rc.LAYER = n.LAYER\n        LEFT JOIN DTS_DATA_ELEMENT e\n               ON e.REPORT_FAMILY = n.REPORT_FAMILY AND e.LAYER = n.LAYER\n              AND e.COLUMN_NAME = n.COLUMN_NAME\n    )\n    , comp_cde AS (   -- CDE-weighted completeness per object\n        SELECT\n            REPORT_FAMILY, LAYER,\n            ROUND(SUM(COMPLETENESS_SCORE * CDE_MULT) / NULLIF(SUM(CDE_MULT), 0), 1)\n                AS COMPLETENESS_CDE_SCORE\n        FROM comp_col\n        GROUP BY REPORT_FAMILY, LAYER\n    )\n    , dq_seeded AS (  -- seeded DAMA sub-dims (accuracy/consistency/validity)\n        SELECT\n            REPORT_FAMILY, LAYER,\n            AVG(IFF(DAMA_SUB_DIM = 'ACCURACY',    PASS_PCT, NULL)) AS ACCURACY_SCORE,\n            AVG(IFF(DAMA_SUB_DIM = 'CONSISTENCY', PASS_PCT, NULL)) AS CONSISTENCY_SCORE,\n            AVG(IFF(DAMA_SUB_DIM = 'VALIDITY',    PASS_PCT, NULL)) AS VALIDITY_SCORE\n        FROM DTS_DQ_RULE_RESULT\n        WHERE SCORE_RUN_DATE <= :RUN_DATE\n        GROUP BY REPORT_FAMILY, LAYER\n    )\n    , dq AS (   -- DQ dimension = mean of available DAMA sub-dims\n        SELECT\n            r.REPORT_FAMILY, r.LAYER,\n            -- average across the non-null DAMA sub-dim scores\n            (\n              COALESCE(cc.COMPLETENESS_CDE_SCORE, 0) * IFF(cc.COMPLETENESS_CDE_SCORE IS NULL,0,1)\n            + COALESCE(d.DQ_UNIQUENESS_SCORE, 0)     * IFF(d.DQ_UNIQUENESS_SCORE IS NULL,0,1)\n            + COALESCE(s.ACCURACY_SCORE, 0)          * IFF(s.ACCURACY_SCORE IS NULL,0,1)\n            + COALESCE(s.CONSISTENCY_SCORE, 0)       * IFF(s.CONSISTENCY_SCORE IS NULL,0,1)\n            + COALESCE(s.VALIDITY_SCORE, 0)          * IFF(s.VALIDITY_SCORE IS NULL,0,1)\n            )\n            / NULLIF(\n                IFF(cc.COMPLETENESS_CDE_SCORE IS NULL,0,1)\n              + IFF(d.DQ_UNIQUENESS_SCORE IS NULL,0,1)\n              + IFF(s.ACCURACY_SCORE IS NULL,0,1)\n              + IFF(s.CONSISTENCY_SCORE IS NULL,0,1)\n              + IFF(s.VALIDITY_SCORE IS NULL,0,1), 0)\n              AS DQ_RAW\n        FROM reg r\n        LEFT JOIN comp_cde  cc ON cc.REPORT_FAMILY = r.REPORT_FAMILY AND cc.LAYER = r.LAYER\n        LEFT JOIN dmf       d  ON d.REPORT_FAMILY  = r.REPORT_FAMILY AND d.LAYER  = r.LAYER\n        LEFT JOIN dq_seeded s  ON s.REPORT_FAMILY  = r.REPORT_FAMILY AND s.LAYER  = r.LAYER\n    )\n    ---------------------------------------------------------------------------\n    -- Observability: avg(freshness, volume) minus open-incident penalties.\n    ---------------------------------------------------------------------------\n    , incidents AS (   -- open incident counts per object (Jira-sourced via SP_DTS_LOAD_JIRA_ISSUES)\n        SELECT REPORT_FAMILY, LAYER,\n               COUNT_IF(SEVERITY = 'P1') AS OPEN_P1,\n               COUNT_IF(SEVERITY = 'P2') AS OPEN_P2,\n               COUNT_IF(SEVERITY = 'P3') AS OPEN_P3\n        FROM DTS_OBSERVABILITY_INCIDENT\n        WHERE STATUS = 'OPEN'\n        GROUP BY REPORT_FAMILY, LAYER\n    )\n    , obs AS (\n        SELECT\n            r.REPORT_FAMILY, r.LAYER,\n            CASE\n              WHEN d.OBS_FRESHNESS_SCORE IS NULL AND d.OBS_VOLUME_SCORE IS NULL THEN NULL\n              ELSE GREATEST(0,\n                   ( COALESCE(d.OBS_FRESHNESS_SCORE,0)*IFF(d.OBS_FRESHNESS_SCORE IS NULL,0,1)\n                   + COALESCE(d.OBS_VOLUME_SCORE,0)*IFF(d.OBS_VOLUME_SCORE IS NULL,0,1) )\n                   / NULLIF(IFF(d.OBS_FRESHNESS_SCORE IS NULL,0,1)+IFF(d.OBS_VOLUME_SCORE IS NULL,0,1),0))\n            END AS OBS_RAW\n        FROM reg r\n        LEFT JOIN dmf d       ON d.REPORT_FAMILY = r.REPORT_FAMILY AND d.LAYER = r.LAYER\n    )\n    ---------------------------------------------------------------------------\n    -- Registry-driven dimensions (NULL when unseeded).\n    ---------------------------------------------------------------------------\n    , own AS (\n        SELECT REPORT_FAMILY, LAYER,\n               (IFF(HAS_OWNER,50,0) + IFF(HAS_STEWARD,50,0)) AS OWNERSHIP_RAW\n        FROM DTS_OWNERSHIP_STEWARDSHIP_REGISTRY\n    )\n    , cls AS (\n        SELECT REPORT_FAMILY, LAYER,\n               CASE WHEN CLASSIFIED_COLUMN_PCT > 0 THEN CLASSIFIED_COLUMN_PCT\n                    WHEN HAS_CLASSIFICATION THEN 100 ELSE 0 END AS CLASSIFICATION_RAW\n        FROM DTS_CLASSIFICATION_REGISTRY\n    )\n    , auth AS (\n        SELECT REPORT_FAMILY, LAYER, IFF(IS_AUTHORITATIVE,100,0) AS AUTH_SOURCE_RAW\n        FROM DTS_SOURCE_CERTIFICATION_REGISTRY\n    )\n    , lin AS (\n        SELECT REPORT_FAMILY, LAYER, MAX(100) AS LINEAGE_RAW\n        FROM DTS_LINEAGE_REGISTRY\n        GROUP BY REPORT_FAMILY, LAYER\n    )\n    , defn AS (\n        SELECT REPORT_FAMILY, LAYER,\n               ROUND(100.0 * AVG(IFF(HAS_DEF,1,0)), 1) AS DEFINITIONS_RAW\n        FROM (\n            SELECT e.REPORT_FAMILY, e.LAYER,\n                   IFF(g.HAS_DEFINITION, TRUE, FALSE) AS HAS_DEF\n            FROM DTS_DATA_ELEMENT e\n            LEFT JOIN DTS_BUSINESS_GLOSSARY g ON g.BUSINESS_TERM = e.BUSINESS_TERM\n            WHERE e.BUSINESS_TERM IS NOT NULL\n        )\n        GROUP BY REPORT_FAMILY, LAYER\n    )\n    , iss AS (   -- Active Issues dimension: 100 minus open Jira incident penalties\n        SELECT r.REPORT_FAMILY, r.LAYER,\n               GREATEST(0, 100 - 10*COALESCE(i.OPEN_P1,0)\n                              - 3*COALESCE(i.OPEN_P2,0)\n                              - 1*COALESCE(i.OPEN_P3,0)) AS ACTIVE_ISSUES_RAW\n        FROM reg r\n        LEFT JOIN incidents i ON i.REPORT_FAMILY = r.REPORT_FAMILY AND i.LAYER = r.LAYER\n    )\n    , usg AS (\n        SELECT REPORT_FAMILY, LAYER,\n               CASE WHEN QUERY_COUNT_30D >= 100 THEN 100\n                    WHEN QUERY_COUNT_30D >= 20  THEN 75\n                    WHEN QUERY_COUNT_30D >= 5   THEN 50\n                    WHEN QUERY_COUNT_30D >= 1   THEN 25\n                    ELSE 0 END AS USAGE_RAW\n        FROM DTS_USAGE_METRICS\n        QUALIFY ROW_NUMBER() OVER (PARTITION BY REPORT_FAMILY, LAYER ORDER BY SCORE_RUN_DATE DESC) = 1\n    )\n    , fb AS (\n        SELECT REPORT_FAMILY, LAYER, ROUND(20.0 * AVG(RATING), 1) AS FEEDBACK_RAW\n        FROM DTS_USER_FEEDBACK\n        GROUP BY REPORT_FAMILY, LAYER\n    )\n    ---------------------------------------------------------------------------\n    -- Unpivot to (family, layer, dim_code, raw_score, confidence_factor).\n    ---------------------------------------------------------------------------\n    , dim_scores AS (\n        SELECT r.REPORT_FAMILY, r.LAYER, 'DQ' AS DIMENSION_CODE,\n               dq.DQ_RAW AS RAW_SCORE, COALESCE(cf.CONFIDENCE_FACTOR,1.0) AS CONFIDENCE_FACTOR\n        FROM reg r LEFT JOIN dq ON dq.REPORT_FAMILY=r.REPORT_FAMILY AND dq.LAYER=r.LAYER\n                   LEFT JOIN cf ON cf.LAYER=r.LAYER\n        UNION ALL\n        SELECT r.REPORT_FAMILY, r.LAYER, 'OBSERVABILITY',\n               obs.OBS_RAW, COALESCE(cf.CONFIDENCE_FACTOR,1.0)\n        FROM reg r LEFT JOIN obs ON obs.REPORT_FAMILY=r.REPORT_FAMILY AND obs.LAYER=r.LAYER\n                   LEFT JOIN cf ON cf.LAYER=r.LAYER\n        UNION ALL\n        SELECT r.REPORT_FAMILY, r.LAYER, 'OWNERSHIP', own.OWNERSHIP_RAW, 1.0\n        FROM reg r LEFT JOIN own ON own.REPORT_FAMILY=r.REPORT_FAMILY AND own.LAYER=r.LAYER\n        UNION ALL\n        SELECT r.REPORT_FAMILY, r.LAYER, 'CLASSIFICATION', cls.CLASSIFICATION_RAW, 1.0\n        FROM reg r LEFT JOIN cls ON cls.REPORT_FAMILY=r.REPORT_FAMILY AND cls.LAYER=r.LAYER\n        UNION ALL\n        SELECT r.REPORT_FAMILY, r.LAYER, 'AUTH_SOURCE', auth.AUTH_SOURCE_RAW, 1.0\n        FROM reg r LEFT JOIN auth ON auth.REPORT_FAMILY=r.REPORT_FAMILY AND auth.LAYER=r.LAYER\n        UNION ALL\n        SELECT r.REPORT_FAMILY, r.LAYER, 'LINEAGE', lin.LINEAGE_RAW, 1.0\n        FROM reg r LEFT JOIN lin ON lin.REPORT_FAMILY=r.REPORT_FAMILY AND lin.LAYER=r.LAYER\n        UNION ALL\n        SELECT r.REPORT_FAMILY, r.LAYER, 'DEFINITIONS', defn.DEFINITIONS_RAW, 1.0\n        FROM reg r LEFT JOIN defn ON defn.REPORT_FAMILY=r.REPORT_FAMILY AND defn.LAYER=r.LAYER\n        UNION ALL\n        SELECT r.REPORT_FAMILY, r.LAYER, 'ACTIVE_ISSUES', iss.ACTIVE_ISSUES_RAW, 1.0\n        FROM reg r LEFT JOIN iss ON iss.REPORT_FAMILY=r.REPORT_FAMILY AND iss.LAYER=r.LAYER\n        UNION ALL\n        SELECT r.REPORT_FAMILY, r.LAYER, 'USAGE', usg.USAGE_RAW, 1.0\n        FROM reg r LEFT JOIN usg ON usg.REPORT_FAMILY=r.REPORT_FAMILY AND usg.LAYER=r.LAYER\n        UNION ALL\n        SELECT r.REPORT_FAMILY, r.LAYER, 'FEEDBACK', fb.FEEDBACK_RAW, 1.0\n        FROM reg r LEFT JOIN fb ON fb.REPORT_FAMILY=r.REPORT_FAMILY AND fb.LAYER=r.LAYER\n    )\n    SELECT\n        REPORT_FAMILY, LAYER, NULL AS COLUMN_NAME, DIMENSION_CODE,\n        ROUND(RAW_SCORE, 1) AS RAW_SCORE,\n        CONFIDENCE_FACTOR,\n        1.0 AS CRITICALITY_MULTIPLIER,\n        IFF(RAW_SCORE IS NOT NULL, TRUE, FALSE) AS IS_MEASURABLE,\n        :RUN_DATE\n    FROM dim_scores;\n\n    n_dim_rows := SQLROWCOUNT;\n\n    ---------------------------------------------------------------------------\n    -- Roll up element/dimension scores to the final dataset trust score.\n    -- Effective score = RAW_SCORE * CONFIDENCE_FACTOR (CDE already folded into\n    -- DQ RAW). Blanks (NULL) count as 0 in the weighted sum, so TRUST_SCORE is\n    -- naturally capped by the measurable ceiling.\n    ---------------------------------------------------------------------------\n    INSERT INTO DTS_DATASET_TRUST_SCORE\n        (REPORT_FAMILY, LAYER, DATASET_FQN, DQ_MEASUREMENT_LAYER, TRUST_SCORE,\n         TRUST_BAND, MEASURABLE_CEILING, MEASURED_SUBTOTAL, FOUNDATIONAL_GAP_FLAG,\n         FOUNDATIONAL_GAPS, SCORE_RUN_DATE)\n    WITH s AS (\n        SELECT eds.REPORT_FAMILY, eds.LAYER, eds.DIMENSION_CODE,\n               eds.RAW_SCORE, eds.CONFIDENCE_FACTOR, eds.IS_MEASURABLE,\n               w.WEIGHT, w.IS_FOUNDATIONAL,\n               eds.RAW_SCORE * eds.CONFIDENCE_FACTOR AS EFFECTIVE_SCORE\n        FROM DTS_ELEMENT_DIMENSION_SCORE eds\n        JOIN DTS_DIMENSION_WEIGHTS w ON w.DIMENSION_CODE = eds.DIMENSION_CODE\n        WHERE eds.SCORE_RUN_DATE = :RUN_DATE\n    )\n    , rollup AS (\n        SELECT\n            REPORT_FAMILY, LAYER,\n            ROUND(SUM(WEIGHT * COALESCE(EFFECTIVE_SCORE,0) / 100.0), 1) AS TRUST_SCORE,\n            SUM(IFF(IS_MEASURABLE, WEIGHT, 0))                          AS MEASURABLE_CEILING,\n            BOOLOR_AGG(\n                (IS_FOUNDATIONAL AND NOT COALESCE(IS_MEASURABLE,FALSE))\n                OR (IS_FOUNDATIONAL AND COALESCE(RAW_SCORE,0)=0)\n            )                                                           AS FOUNDATIONAL_GAP_FLAG,\n            ARRAY_AGG(IFF(IS_FOUNDATIONAL AND COALESCE(RAW_SCORE,0)=0, DIMENSION_CODE, NULL))\n                                                                        AS GAPS_RAW\n        FROM s\n        GROUP BY REPORT_FAMILY, LAYER\n    )\n    SELECT\n        ru.REPORT_FAMILY, ru.LAYER, reg.DATASET_FQN, reg.DQ_MEASUREMENT_LAYER,\n        ru.TRUST_SCORE,\n        COALESCE(b.BAND_NAME, 'AT_RISK') AS TRUST_BAND,\n        ru.MEASURABLE_CEILING,\n        ru.TRUST_SCORE AS MEASURED_SUBTOTAL,\n        ru.FOUNDATIONAL_GAP_FLAG,\n        ARRAY_COMPACT(ru.GAPS_RAW) AS FOUNDATIONAL_GAPS,\n        :RUN_DATE\n    FROM rollup ru\n    JOIN DTS_DATASET_REGISTRY reg\n      ON reg.REPORT_FAMILY = ru.REPORT_FAMILY AND reg.LAYER = ru.LAYER\n    -- Pick the highest band whose MIN_SCORE the score meets. Using MIN_SCORE\n    -- only (not BETWEEN) avoids gaps for fractional FLOAT scores, e.g. 89.5.\n    LEFT JOIN DTS_TRUST_BANDS b\n      ON ru.TRUST_SCORE >= b.MIN_SCORE\n    QUALIFY ROW_NUMBER() OVER (\n        PARTITION BY ru.REPORT_FAMILY, ru.LAYER\n        ORDER BY b.MIN_SCORE DESC\n    ) = 1;\n\n    n_score_rows := SQLROWCOUNT;\n\n    RETURN OBJECT_CONSTRUCT('run_date', :RUN_DATE,\n                            'dimension_rows', :n_dim_rows,\n                            'dataset_rows', :n_score_rows);\nEND;\n$$;").collect()

In [ ]:
COMMENT ON PROCEDURE SP_DTS_COMPUTE_SCORES(DATE) IS
    'Compute v2 trust scores: element DQ (CDE-weighted) + Obs x confidence + registry dims -> weighted rollup -> ceiling/gap/band. Writes DTS_ELEMENT_DIMENSION_SCORE + DTS_DATASET_TRUST_SCORE.';

-- Convenience consumer view: latest run per object.
CREATE OR REPLACE VIEW DTS_VW_DATASET_TRUST_SCORE_LATEST AS
SELECT *
FROM DTS_DATASET_TRUST_SCORE
QUALIFY ROW_NUMBER() OVER (PARTITION BY REPORT_FAMILY, LAYER ORDER BY SCORE_RUN_DATE DESC) = 1;

## 9b · Element-level score detail view (`DTS_VW_ELEMENT_TRUST_DETAIL`)

Per (family, layer, column) score for each of the 10 dimensions (DQ/Definitions/Classification per element, the rest inherited) + `ELEMENT_SCORE`. Powers the app's Dataset Detail element matrix. Read-only view; returns rows once `SP_DTS_COMPUTE_SCORES()` has run.

In [ ]:
-------------------------------------------------------------------------------
-- ddl/32_dts_element_detail.sql
-- v2 Data Trust Score -- element (column) level score detail.
--
-- DTS_VW_ELEMENT_TRUST_DETAIL exposes, per (report_family, layer, column), a
-- score for each of the 10 trust dimensions -- the drill-down behind the
-- dataset score (the app's "Score - STG_*" matrix from the framework workbook).
--
--   Per-element (true column-grain data where it exists):
--     DQ            -- per-column completeness (NULL_COUNT vs row count) x layer
--                      confidence factor; falls back to the dataset DQ score
--                      when the column has no DMF reading.
--     DEFINITIONS   -- 100 if the column's BUSINESS_TERM has an approved
--                      glossary definition, else 0.
--     CLASSIFICATION-- 100 if the column carries a PII_TAG, else 0.
--     CDE flag / CRITICALITY_MULTIPLIER from DTS_DATA_ELEMENT.
--
--   Inherited from the dataset (table-grain today -- same value for every
--   column of the object): OBSERVABILITY, OWNERSHIP, AUTH_SOURCE, LINEAGE,
--   ACTIVE_ISSUES, USAGE, FEEDBACK. Taken from DTS_ELEMENT_DIMENSION_SCORE
--   (COLUMN_NAME IS NULL) latest run as RAW_SCORE x CONFIDENCE_FACTOR.
--
--   ELEMENT_SCORE /100 = weighted sum of the row's dimension scores using
--   DTS_DIMENSION_WEIGHTS (weights sum to 100) -- mirrors the workbook's
--   "Element Score /100" column.
--
-- Read-only: derives everything from existing objects; does NOT modify the
-- scoring engine or the dataset rollup. Deploy AFTER ddl/30 (needs
-- DTS_ELEMENT_DIMENSION_SCORE populated by SP_DTS_COMPUTE_SCORES at run time).
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

CREATE OR REPLACE VIEW DTS_VW_ELEMENT_TRUST_DETAIL AS
WITH latest_run AS (
    SELECT MAX(SCORE_RUN_DATE) AS RUN_DATE FROM DTS_ELEMENT_DIMENSION_SCORE
)
, reg AS (
    SELECT REPORT_FAMILY, LAYER
    FROM DTS_DATASET_REGISTRY
    WHERE IS_IN_SCOPE = TRUE
)
-- Table-grain dimension EFFECTIVE scores (RAW x CF) for the latest run,
-- pivoted wide so each element can inherit them.
, dim AS (
    SELECT
        REPORT_FAMILY, LAYER,
        MAX(IFF(DIMENSION_CODE = 'DQ',            RAW_SCORE * CONFIDENCE_FACTOR, NULL)) AS DQ_DS,
        MAX(IFF(DIMENSION_CODE = 'OBSERVABILITY', RAW_SCORE * CONFIDENCE_FACTOR, NULL)) AS OBSERVABILITY_DS,
        MAX(IFF(DIMENSION_CODE = 'OWNERSHIP',     RAW_SCORE * CONFIDENCE_FACTOR, NULL)) AS OWNERSHIP_DS,
        MAX(IFF(DIMENSION_CODE = 'CLASSIFICATION',RAW_SCORE * CONFIDENCE_FACTOR, NULL)) AS CLASSIFICATION_DS,
        MAX(IFF(DIMENSION_CODE = 'AUTH_SOURCE',   RAW_SCORE * CONFIDENCE_FACTOR, NULL)) AS AUTH_SOURCE_DS,
        MAX(IFF(DIMENSION_CODE = 'LINEAGE',       RAW_SCORE * CONFIDENCE_FACTOR, NULL)) AS LINEAGE_DS,
        MAX(IFF(DIMENSION_CODE = 'DEFINITIONS',   RAW_SCORE * CONFIDENCE_FACTOR, NULL)) AS DEFINITIONS_DS,
        MAX(IFF(DIMENSION_CODE = 'ACTIVE_ISSUES', RAW_SCORE * CONFIDENCE_FACTOR, NULL)) AS ACTIVE_ISSUES_DS,
        MAX(IFF(DIMENSION_CODE = 'USAGE',         RAW_SCORE * CONFIDENCE_FACTOR, NULL)) AS USAGE_DS,
        MAX(IFF(DIMENSION_CODE = 'FEEDBACK',      RAW_SCORE * CONFIDENCE_FACTOR, NULL)) AS FEEDBACK_DS
    FROM DTS_ELEMENT_DIMENSION_SCORE
    WHERE COLUMN_NAME IS NULL
      AND SCORE_RUN_DATE = (SELECT RUN_DATE FROM latest_run)
    GROUP BY REPORT_FAMILY, LAYER
)
, cf AS (
    SELECT LAYER, CONFIDENCE_FACTOR
    FROM DTS_DQ_CONFIDENCE_FACTOR
    WHERE LAYER IS NOT NULL
)
, rowcnt AS (   -- object row count (for per-column completeness denominator)
    -- DTS_VW_DMF_DIMENSION_SCORES is grained by (family, layer, measurement
    -- layer); collapse to one row per (family, layer) so the join to elements
    -- can never fan out if a second measurement layer ever appears.
    SELECT REPORT_FAMILY, LAYER, MAX(ROW_COUNT_V) AS ROW_COUNT_V
    FROM DTS_VW_DMF_DIMENSION_SCORES
    GROUP BY REPORT_FAMILY, LAYER
)
, nullcol AS (  -- per-column NULL_COUNT, when the DMF has measured it
    SELECT REPORT_FAMILY, LAYER, COLUMN_NAME, VALUE AS NULL_CNT
    FROM DTS_VW_DMF_LATEST_MEASUREMENTS
    WHERE METRIC_NAME = 'SNOWFLAKE.CORE.NULL_COUNT'
      AND COLUMN_NAME IS NOT NULL
)
, elem AS (
    SELECT
        e.REPORT_FAMILY, e.LAYER, e.COLUMN_NAME,
        e.IS_KEY, e.IS_CDE, e.CRITICALITY_MULTIPLIER, e.PII_TAG, e.BUSINESS_TERM,
        -- DQ: per-column completeness x CF, else inherit dataset DQ effective.
        CASE
            WHEN nc.NULL_CNT IS NOT NULL AND rc.ROW_COUNT_V IS NOT NULL AND rc.ROW_COUNT_V > 0
                THEN ROUND(GREATEST(0, 100 - 100.0 * nc.NULL_CNT / rc.ROW_COUNT_V)
                           * COALESCE(cf.CONFIDENCE_FACTOR, 1.0), 1)
            ELSE dim.DQ_DS
        END                                                       AS DQ_EL,
        IFF(g.HAS_DEFINITION, 100, 0)                             AS DEFINITIONS_EL,
        IFF(e.PII_TAG IS NOT NULL, 100, 0)                        AS CLASSIFICATION_EL,
        dim.OBSERVABILITY_DS                                      AS OBSERVABILITY_EL,
        dim.OWNERSHIP_DS                                          AS OWNERSHIP_EL,
        dim.AUTH_SOURCE_DS                                        AS AUTH_SOURCE_EL,
        dim.LINEAGE_DS                                            AS LINEAGE_EL,
        dim.ACTIVE_ISSUES_DS                                      AS ACTIVE_ISSUES_EL,
        dim.USAGE_DS                                              AS USAGE_EL,
        dim.FEEDBACK_DS                                           AS FEEDBACK_EL
    FROM DTS_DATA_ELEMENT e
    JOIN reg r        ON r.REPORT_FAMILY = e.REPORT_FAMILY AND r.LAYER = e.LAYER
    LEFT JOIN dim     ON dim.REPORT_FAMILY = e.REPORT_FAMILY AND dim.LAYER = e.LAYER
    LEFT JOIN cf      ON cf.LAYER = e.LAYER
    LEFT JOIN rowcnt rc ON rc.REPORT_FAMILY = e.REPORT_FAMILY AND rc.LAYER = e.LAYER
    LEFT JOIN nullcol nc ON nc.REPORT_FAMILY = e.REPORT_FAMILY AND nc.LAYER = e.LAYER
                        AND nc.COLUMN_NAME = e.COLUMN_NAME
    LEFT JOIN DTS_BUSINESS_GLOSSARY g ON g.BUSINESS_TERM = e.BUSINESS_TERM
)
-- Unpivot to (column, dimension, score) so ELEMENT_SCORE is computed from the
-- weights table (config-driven) rather than hardcoded coefficients.
, elem_long AS (
    SELECT REPORT_FAMILY, LAYER, COLUMN_NAME, 'DQ'             AS DIMENSION_CODE, DQ_EL            AS SC FROM elem
    UNION ALL SELECT REPORT_FAMILY, LAYER, COLUMN_NAME, 'OBSERVABILITY',  OBSERVABILITY_EL  FROM elem
    UNION ALL SELECT REPORT_FAMILY, LAYER, COLUMN_NAME, 'OWNERSHIP',      OWNERSHIP_EL      FROM elem
    UNION ALL SELECT REPORT_FAMILY, LAYER, COLUMN_NAME, 'CLASSIFICATION', CLASSIFICATION_EL FROM elem
    UNION ALL SELECT REPORT_FAMILY, LAYER, COLUMN_NAME, 'AUTH_SOURCE',    AUTH_SOURCE_EL    FROM elem
    UNION ALL SELECT REPORT_FAMILY, LAYER, COLUMN_NAME, 'LINEAGE',        LINEAGE_EL        FROM elem
    UNION ALL SELECT REPORT_FAMILY, LAYER, COLUMN_NAME, 'DEFINITIONS',    DEFINITIONS_EL    FROM elem
    UNION ALL SELECT REPORT_FAMILY, LAYER, COLUMN_NAME, 'ACTIVE_ISSUES',  ACTIVE_ISSUES_EL  FROM elem
    UNION ALL SELECT REPORT_FAMILY, LAYER, COLUMN_NAME, 'USAGE',          USAGE_EL          FROM elem
    UNION ALL SELECT REPORT_FAMILY, LAYER, COLUMN_NAME, 'FEEDBACK',       FEEDBACK_EL       FROM elem
)
, elem_score AS (
    SELECT
        el.REPORT_FAMILY, el.LAYER, el.COLUMN_NAME,
        -- Self-normalize over the weights actually present + non-null so a
        -- missing/NULL dimension can't silently understate the score, and the
        -- scale stays correct if the weights table ever drifts from summing 100.
        ROUND(SUM(w.WEIGHT * el.SC)
              / NULLIF(SUM(IFF(el.SC IS NULL, 0, w.WEIGHT)), 0), 1) AS ELEMENT_SCORE
    FROM elem_long el
    JOIN DTS_DIMENSION_WEIGHTS w ON w.DIMENSION_CODE = el.DIMENSION_CODE
    GROUP BY el.REPORT_FAMILY, el.LAYER, el.COLUMN_NAME
)
SELECT
    elem.REPORT_FAMILY, elem.LAYER, elem.COLUMN_NAME,
    elem.IS_KEY, elem.IS_CDE, elem.CRITICALITY_MULTIPLIER, elem.PII_TAG, elem.BUSINESS_TERM,
    elem.DQ_EL, elem.OBSERVABILITY_EL, elem.OWNERSHIP_EL, elem.CLASSIFICATION_EL,
    elem.AUTH_SOURCE_EL, elem.LINEAGE_EL, elem.DEFINITIONS_EL, elem.ACTIVE_ISSUES_EL,
    elem.USAGE_EL, elem.FEEDBACK_EL,
    es.ELEMENT_SCORE
FROM elem
LEFT JOIN elem_score es
       ON es.REPORT_FAMILY = elem.REPORT_FAMILY
      AND es.LAYER = elem.LAYER
      AND es.COLUMN_NAME = elem.COLUMN_NAME;

COMMENT ON VIEW DTS_VW_ELEMENT_TRUST_DETAIL IS
    'Per (report_family, layer, column) score for each of the 10 trust dimensions: DQ/Definitions/Classification/CDE per element, the rest inherited from the dataset; ELEMENT_SCORE = weighted sum. Drill-down behind the dataset trust score.';

## 10 · Seed registries + metadata  _(run before the CALLs below)_

In [ ]:
-------------------------------------------------------------------------------
-- ddl/40_dts_seed_metadata.sql
-- v2 Data Trust Score -- seed the registries + lineage + interim signals.
--
-- Populates the governance/metadata tables for the 5 scored objects:
--   POSITION_REPORT: INT, DW, PUBL      TRENDED_REPORT: DW, PUBL (INT skipped --
--   empty at rest; DW is source-of-record).
--
-- Child tables carry DATASET_ID; we look it up from DTS_DATASET_REGISTRY by the
-- (REPORT_FAMILY, LAYER) natural key so IDENTITY values don't have to be known.
--
-- Run AFTER ddl/00-04. Then clone + attach DMFs (ddl/20, 21), let DMFs measure,
-- and CALL SP_DTS_COMPUTE_SCORES() (ddl/30).
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

-------------------------------------------------------------------------------
-- 1. DTS_DATASET_REGISTRY -- the 5 scored objects.
-------------------------------------------------------------------------------
TRUNCATE TABLE DTS_DATASET_REGISTRY;

INSERT INTO DTS_DATASET_REGISTRY
    (REPORT_FAMILY, LAYER, DATABASE_NAME, SCHEMA_NAME, OBJECT_NAME, OBJECT_TYPE,
     DATASET_FQN, DQ_MEASUREMENT_LAYER, IS_VIEW, FRESHNESS_COLUMN, KEY_COLUMNS,
     MARKETPLACE_STATUS, IS_IN_SCOPE, CLONE_EXISTS)
SELECT 'POSITION_REPORT','INT','EDLE_INT_DB','PNC_WORKDAY01','STG_POSITION_REPORT','TABLE',
     'EDLE_INT_DB.PNC_WORKDAY01.STG_POSITION_REPORT','BRONZE',FALSE,'LOADDATE',
     ARRAY_CONSTRUCT('POSITION_ID','REPORT_EFFECTIVE_DATE','REPORT_ENTRY_DATE'),'INTERNAL',TRUE,FALSE
UNION ALL
SELECT 'POSITION_REPORT','DW','EDLE_DW_DB','PNC_DATA','DT_POSITION_REPORT','TABLE',
     'EDLE_DW_DB.PNC_DATA.DT_POSITION_REPORT','SILVER',FALSE,'LOADDATE',
     ARRAY_CONSTRUCT('POSITION_ID','REPORT_EFFECTIVE_DATE','REPORT_ENTRY_DATE'),'INTERNAL',TRUE,FALSE
UNION ALL
SELECT 'POSITION_REPORT','PUBL','EDLE_PUBL_DB','PNC_ANALYTICS','STG_POSITION_REPORT_VW','VIEW',
     'EDLE_PUBL_DB.PNC_ANALYTICS.STG_POSITION_REPORT_VW','GOLD',TRUE,'Load Date date and timestamp',
     ARRAY_CONSTRUCT('Position ID','Report Effective Date','Report Entry Date'),'INTERNAL',TRUE,FALSE
UNION ALL
SELECT 'TRENDED_REPORT','DW','EDLE_DW_DB','PNC_DATA','DT_TRENDED_REPORT','TABLE',
     'EDLE_DW_DB.PNC_DATA.DT_TRENDED_REPORT','SILVER',FALSE,'LOADDATE',
     ARRAY_CONSTRUCT('BUSINESS_PROCESS_WID','EFFECTIVEDATE','EMPLOYEEID','RECORDTYPE'),'INTERNAL',TRUE,FALSE
UNION ALL
SELECT 'TRENDED_REPORT','PUBL','EDLE_PUBL_DB','PNC_ANALYTICS','STG_TRENDED_REPORT_VW','VIEW',
     'EDLE_PUBL_DB.PNC_ANALYTICS.STG_TRENDED_REPORT_VW','GOLD',TRUE,'Load Date date and timestamp',
     ARRAY_CONSTRUCT('Business Process WID','Effective Date','Employee ID','Record Type'),'INTERNAL',TRUE,FALSE;

-------------------------------------------------------------------------------
-- 2. DTS_DATA_ELEMENT -- key-column inventory + CDE flags.
--    Identifier keys (POSITION_ID, EMPLOYEEID, BUSINESS_PROCESS_WID) are CDE
--    (2.0x). Date keys are keys but not CDE. Additional CDE columns can be
--    added later from the WPII_/WSPII_ PII tag list.
-------------------------------------------------------------------------------
DELETE FROM DTS_DATA_ELEMENT;

INSERT INTO DTS_DATA_ELEMENT
    (DATASET_ID, REPORT_FAMILY, LAYER, COLUMN_NAME, IS_KEY, IS_CDE,
     CRITICALITY_MULTIPLIER, PII_TAG, BUSINESS_TERM)
SELECT r.DATASET_ID, v.column1, v.column2, v.column3, v.column4, v.column5,
       v.column6, v.column7, v.column8
FROM VALUES
    -- POSITION family (same key set at each layer)
    ('POSITION_REPORT','INT','POSITION_ID',TRUE,TRUE,2.0,'WPII_ALPHANUM_ID','Position ID'),
    ('POSITION_REPORT','INT','REPORT_EFFECTIVE_DATE',TRUE,FALSE,1.0,NULL,'Report Effective Date'),
    ('POSITION_REPORT','INT','REPORT_ENTRY_DATE',TRUE,FALSE,1.0,NULL,'Report Entry Date'),
    ('POSITION_REPORT','DW','POSITION_ID',TRUE,TRUE,2.0,'WPII_ALPHANUM_ID','Position ID'),
    ('POSITION_REPORT','DW','REPORT_EFFECTIVE_DATE',TRUE,FALSE,1.0,NULL,'Report Effective Date'),
    ('POSITION_REPORT','DW','REPORT_ENTRY_DATE',TRUE,FALSE,1.0,NULL,'Report Entry Date'),
    ('POSITION_REPORT','PUBL','Position ID',TRUE,TRUE,2.0,'WPII_ALPHANUM_ID','Position ID'),
    ('POSITION_REPORT','PUBL','Report Effective Date',TRUE,FALSE,1.0,NULL,'Report Effective Date'),
    ('POSITION_REPORT','PUBL','Report Entry Date',TRUE,FALSE,1.0,NULL,'Report Entry Date'),
    -- TRENDED family (DW, PUBL)
    ('TRENDED_REPORT','DW','BUSINESS_PROCESS_WID',TRUE,TRUE,2.0,'WPII_ALPHANUM_ID','Business Process WID'),
    ('TRENDED_REPORT','DW','EFFECTIVEDATE',TRUE,FALSE,1.0,NULL,'Effective Date'),
    ('TRENDED_REPORT','DW','EMPLOYEEID',TRUE,TRUE,2.0,'WPII_ALPHANUM_ID','Employee ID'),
    ('TRENDED_REPORT','DW','RECORDTYPE',TRUE,FALSE,1.0,NULL,'Record Type'),
    ('TRENDED_REPORT','PUBL','Business Process WID',TRUE,TRUE,2.0,'WPII_ALPHANUM_ID','Business Process WID'),
    ('TRENDED_REPORT','PUBL','Effective Date',TRUE,FALSE,1.0,NULL,'Effective Date'),
    ('TRENDED_REPORT','PUBL','Employee ID',TRUE,TRUE,2.0,'WPII_ALPHANUM_ID','Employee ID'),
    ('TRENDED_REPORT','PUBL','Record Type',TRUE,FALSE,1.0,NULL,'Record Type') v
JOIN DTS_DATASET_REGISTRY r
  ON r.REPORT_FAMILY = v.column1 AND r.LAYER = v.column2;

-------------------------------------------------------------------------------
-- 3. DTS_BUSINESS_GLOSSARY -- definitions for the seeded business terms.
-------------------------------------------------------------------------------
DELETE FROM DTS_BUSINESS_GLOSSARY;

INSERT INTO DTS_BUSINESS_GLOSSARY (BUSINESS_TERM, DEFINITION, HAS_DEFINITION, STEWARD, SOURCE_SYSTEM)
VALUES
    ('Position ID','Unique identifier of a Workday position.',TRUE,'P&C Data Steward','COLLIBRA'),
    ('Report Effective Date','Business-effective date of the position snapshot.',TRUE,'P&C Data Steward','COLLIBRA'),
    ('Report Entry Date','Date the record entered the report.',TRUE,'P&C Data Steward','COLLIBRA'),
    ('Business Process WID','Workday business-process instance identifier.',TRUE,'P&C Data Steward','COLLIBRA'),
    ('Effective Date','Effective date of the trended record.',TRUE,'P&C Data Steward','COLLIBRA'),
    ('Employee ID','Unique Workday employee identifier.',TRUE,'P&C Data Steward','COLLIBRA'),
    ('Record Type',NULL,FALSE,NULL,'COLLIBRA');

-- intentionally undefined (coverage < 100%)

-------------------------------------------------------------------------------
-- 4. Foundational dims: ownership, source certification, classification.
-------------------------------------------------------------------------------
DELETE FROM DTS_OWNERSHIP_STEWARDSHIP_REGISTRY;

INSERT INTO DTS_OWNERSHIP_STEWARDSHIP_REGISTRY
    (DATASET_ID, REPORT_FAMILY, LAYER, DATA_OWNER, DATA_STEWARD, TECHNICAL_OWNER, HAS_OWNER, HAS_STEWARD)
SELECT r.DATASET_ID, r.REPORT_FAMILY, r.LAYER,
       'P&C Analytics Lead', 'P&C Data Steward', 'EDL Platform Team', TRUE, TRUE
FROM DTS_DATASET_REGISTRY r;

DELETE FROM DTS_SOURCE_CERTIFICATION_REGISTRY;

INSERT INTO DTS_SOURCE_CERTIFICATION_REGISTRY
    (DATASET_ID, REPORT_FAMILY, LAYER, SOURCE_SYSTEM, IS_AUTHORITATIVE, CERTIFIED_BY, CERTIFIED_DATE)
SELECT r.DATASET_ID, r.REPORT_FAMILY, r.LAYER, 'WORKDAY', TRUE, 'EDAI Governance', CURRENT_DATE()
FROM DTS_DATASET_REGISTRY r;

DELETE FROM DTS_CLASSIFICATION_REGISTRY;

INSERT INTO DTS_CLASSIFICATION_REGISTRY
    (DATASET_ID, REPORT_FAMILY, LAYER, HAS_CLASSIFICATION, SENSITIVITY_LEVEL, PII_PRESENT, CLASSIFIED_COLUMN_PCT)
SELECT r.DATASET_ID, r.REPORT_FAMILY, r.LAYER, TRUE, 'CONFIDENTIAL', TRUE, 100
FROM DTS_DATASET_REGISTRY r;

-------------------------------------------------------------------------------
-- 5. DTS_LINEAGE_REGISTRY -- the two confirmed chains (from edl-pnc-repo).
-------------------------------------------------------------------------------
DELETE FROM DTS_LINEAGE_REGISTRY;

INSERT INTO DTS_LINEAGE_REGISTRY
    (REPORT_FAMILY, LAYER, OBJECT_FQN, UPSTREAM_FQN, DOWNSTREAM_FQN, TRANSFORM_TYPE, HAS_UPSTREAM, HAS_DOWNSTREAM)
VALUES
    -- POSITION chain: S3 -> INT (COPY) -> DW (INSERT) -> PUBL (VIEW)
    ('POSITION_REPORT','INT','EDLE_INT_DB.PNC_WORKDAY01.STG_POSITION_REPORT',
     'S3://edl-pnc-workday', 'EDLE_DW_DB.PNC_DATA.DT_POSITION_REPORT','COPY',TRUE,TRUE),
    ('POSITION_REPORT','DW','EDLE_DW_DB.PNC_DATA.DT_POSITION_REPORT',
     'EDLE_INT_DB.PNC_WORKDAY01.STG_POSITION_REPORT','EDLE_PUBL_DB.PNC_ANALYTICS.STG_POSITION_REPORT_VW','INSERT',TRUE,TRUE),
    ('POSITION_REPORT','PUBL','EDLE_PUBL_DB.PNC_ANALYTICS.STG_POSITION_REPORT_VW',
     'EDLE_DW_DB.PNC_DATA.DT_POSITION_REPORT', NULL,'VIEW',TRUE,FALSE),
    -- TRENDED chain (DW is source-of-record; INT omitted)
    ('TRENDED_REPORT','DW','EDLE_DW_DB.PNC_DATA.DT_TRENDED_REPORT',
     'EDLE_INT_DB.PNC_WORKDAY01.STG_TRENDED_REPORT','EDLE_PUBL_DB.PNC_ANALYTICS.STG_TRENDED_REPORT_VW','MERGE',TRUE,TRUE),
    ('TRENDED_REPORT','PUBL','EDLE_PUBL_DB.PNC_ANALYTICS.STG_TRENDED_REPORT_VW',
     'EDLE_DW_DB.PNC_DATA.DT_TRENDED_REPORT', NULL,'VIEW',TRUE,FALSE);

-------------------------------------------------------------------------------
-- 6. Interim DQ rule results (IDQ seeds for accuracy/consistency/validity).
--    Completeness + uniqueness come from live DMFs; these three are interim
--    until native rules exist. PASS_PCT feeds the DQ dimension blend.
-------------------------------------------------------------------------------
DELETE FROM DTS_DQ_RULE_RESULT WHERE SOURCE_SYSTEM = 'IDQ';

INSERT INTO DTS_DQ_RULE_RESULT
    (REPORT_FAMILY, LAYER, COLUMN_NAME, DAMA_SUB_DIM, RULE_NAME, IS_DMF_AUTOMATED,
     PASS_PCT, MEASURED_VALUE, THRESHOLD, STATUS, SOURCE_SYSTEM, SCORE_RUN_DATE)
SELECT r.REPORT_FAMILY, r.LAYER, NULL, v.column1, v.column2, FALSE,
       v.column3, v.column3, 95, IFF(v.column3 >= 95,'PASS','WARN'), 'IDQ', CURRENT_DATE()
FROM VALUES
    ('ACCURACY',   'IDQ accuracy profile',    92),
    ('CONSISTENCY','IDQ cross-field consistency', 96),
    ('VALIDITY',   'IDQ domain/format validity',  90) v
CROSS JOIN DTS_DATASET_REGISTRY r;

-------------------------------------------------------------------------------
-- 7. Usage + feedback (minimal seeds; Usage can later come from ACCESS_HISTORY).
-------------------------------------------------------------------------------
DELETE FROM DTS_USAGE_METRICS;

INSERT INTO DTS_USAGE_METRICS (REPORT_FAMILY, LAYER, QUERY_COUNT_30D, DISTINCT_USERS_30D, LAST_QUERIED_AT, SCORE_RUN_DATE)
SELECT r.REPORT_FAMILY, r.LAYER,
       IFF(r.LAYER='PUBL', 120, IFF(r.LAYER='DW', 40, 8)),
       IFF(r.LAYER='PUBL', 15, IFF(r.LAYER='DW', 6, 2)),
       CURRENT_TIMESTAMP(), CURRENT_DATE()
FROM DTS_DATASET_REGISTRY r;

DELETE FROM DTS_USER_FEEDBACK;

INSERT INTO DTS_USER_FEEDBACK (REPORT_FAMILY, LAYER, RATING, COMMENT_TEXT, SUBMITTED_BY)
SELECT r.REPORT_FAMILY, r.LAYER, 4, 'Reliable for reporting.', 'analyst@wbd.com'
FROM DTS_DATASET_REGISTRY r WHERE r.LAYER = 'PUBL';

-------------------------------------------------------------------------------
-- Verify seed counts.
-------------------------------------------------------------------------------
SELECT 'DATASET_REGISTRY' AS tbl, COUNT(*) AS n FROM DTS_DATASET_REGISTRY
UNION ALL SELECT 'DATA_ELEMENT', COUNT(*) FROM DTS_DATA_ELEMENT
UNION ALL SELECT 'LINEAGE', COUNT(*) FROM DTS_LINEAGE_REGISTRY
UNION ALL SELECT 'DQ_RULE_RESULT', COUNT(*) FROM DTS_DQ_RULE_RESULT
ORDER BY tbl;

## 10b · Jira -> Active Issues  (loader + sample)

Creates `SP_DTS_LOAD_JIRA_ISSUES` and loads a **sample** payload into `DTS_OBSERVABILITY_INCIDENT` so the Active Issues dimension is exercised end-to-end. Replace the sample `PARSE_JSON(...)` with the real EDA issues array once the Atlassian MCP is authorized. This step also re-runs `SP_DTS_COMPUTE_SCORES()`.

In [ ]:
-------------------------------------------------------------------------------
-- ddl/45_dts_jira_incidents.sql
-- v2 Data Trust Score -- Jira -> Active Issues dimension.
--
-- The scoring engine (ddl/30) already derives the Active Issues dimension from
-- open rows in DTS_OBSERVABILITY_INCIDENT:
--     ACTIVE_ISSUES = 100 - 10*openP1 - 3*openP2 - 1*openP3   (per family x layer)
-- This file supplies the missing loader (SP_DTS_LOAD_JIRA_ISSUES) plus a sample
-- payload so the dimension is exercised end-to-end before the Atlassian MCP is
-- connected.
--
-- Mapping (matches the agreed design):
--   REPORT_FAMILY  <- the issue's "dataset:<FAMILY>" label (project EDA, board 16274)
--   SEVERITY       <- Jira priority: Highest/High -> P1, Medium -> P2, else P3
--   STATUS         <- statusCategory.key: 'done' -> RESOLVED, else OPEN
--   LAYER          <- expanded to every in-scope layer of the family (issues carry no layer)
--   INCIDENT_TYPE  <- 'JIRA' (so a reload cleanly replaces the prior Jira set)
--
-- Once the MCP is authorized, replace the sample PARSE_JSON payload at the bottom
-- with the array returned by the Jira search tool -- no other change needed.
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

In [ ]:
session.sql("-------------------------------------------------------------------------------\n-- 1. Loader: map a Jira issues array (REST/MCP shape) into DTS_OBSERVABILITY_INCIDENT.\n--    Re-runnable: clears prior INCIDENT_TYPE='JIRA' rows first.\n-------------------------------------------------------------------------------\nCREATE OR REPLACE PROCEDURE SP_DTS_LOAD_JIRA_ISSUES(ISSUES VARIANT)\nRETURNS VARIANT\nLANGUAGE SQL\nEXECUTE AS CALLER\nAS\n$$\nDECLARE\n    loaded INTEGER DEFAULT 0;\nBEGIN\n    DELETE FROM DTS_OBSERVABILITY_INCIDENT WHERE INCIDENT_TYPE = 'JIRA';\n\n    INSERT INTO DTS_OBSERVABILITY_INCIDENT\n        (REPORT_FAMILY, LAYER, INCIDENT_TYPE, SEVERITY, STATUS, DETAIL, DETECTED_AT)\n    WITH iss AS (\n        SELECT f.value AS j\n        FROM   LATERAL FLATTEN(input => :ISSUES) f\n    )\n    , lab AS (\n        SELECT\n            i.j:key::STRING                                        AS issue_key,\n            i.j:fields.summary::STRING                             AS summary,\n            i.j:fields.priority.name::STRING                       AS priority,\n            COALESCE(i.j:fields.status.statusCategory.key::STRING, 'new') AS status_cat,\n            REPLACE(l.value::STRING, 'dataset:', '')               AS report_family\n        FROM iss i,\n             LATERAL FLATTEN(input => i.j:fields.labels) l\n        WHERE l.value::STRING ILIKE 'dataset:%'\n    )\n    SELECT\n        reg.REPORT_FAMILY,\n        reg.LAYER,\n        'JIRA',\n        CASE WHEN lab.priority IN ('Highest','High') THEN 'P1'\n             WHEN lab.priority = 'Medium'            THEN 'P2'\n             ELSE 'P3' END,\n        IFF(lab.status_cat = 'done', 'RESOLVED', 'OPEN'),\n        lab.issue_key || ' - ' || COALESCE(lab.summary, ''),\n        CURRENT_TIMESTAMP()\n    FROM lab\n    JOIN DTS_DATASET_REGISTRY reg\n      ON reg.REPORT_FAMILY = lab.report_family\n     AND reg.IS_IN_SCOPE = TRUE;\n\n    loaded := SQLROWCOUNT;\n    RETURN OBJECT_CONSTRUCT('incident_type', 'JIRA', 'rows_loaded', :loaded);\nEND;\n$$;").collect()

In [ ]:
COMMENT ON PROCEDURE SP_DTS_LOAD_JIRA_ISSUES(VARIANT) IS
    'Map a Jira issues array (REST/MCP shape) into DTS_OBSERVABILITY_INCIDENT; dataset:<FAMILY> label -> REPORT_FAMILY, priority -> P1/P2/P3, expanded to all in-scope layers. Feeds the Active Issues dimension.';

-------------------------------------------------------------------------------
-- 2. SAMPLE payload (remove once the MCP returns real EDA issues).
--    JQL to use with the Atlassian MCP once authorized:
--      project = EDA AND statusCategory != Done
--      AND labels IN ("dataset:POSITION_REPORT","dataset:TRENDED_REPORT")
-------------------------------------------------------------------------------
CALL SP_DTS_LOAD_JIRA_ISSUES(PARSE_JSON('[
  {"key":"EDA-1012","fields":{"summary":"Headcount mismatch vs Workday source","priority":{"name":"High"},   "status":{"statusCategory":{"key":"indeterminate"}},"labels":["dataset:POSITION_REPORT"]}},
  {"key":"EDA-1031","fields":{"summary":"Null POSITION_ID in latest load",     "priority":{"name":"Highest"},"status":{"statusCategory":{"key":"new"}},          "labels":["dataset:POSITION_REPORT"]}},
  {"key":"EDA-1044","fields":{"summary":"Report label typo",                   "priority":{"name":"Low"},    "status":{"statusCategory":{"key":"new"}},          "labels":["dataset:POSITION_REPORT"]}},
  {"key":"EDA-1050","fields":{"summary":"Trended snapshot 2 days late",         "priority":{"name":"Medium"}, "status":{"statusCategory":{"key":"indeterminate"}},"labels":["dataset:TRENDED_REPORT"]}},
  {"key":"EDA-1077","fields":{"summary":"Duplicate Business Process WID",       "priority":{"name":"High"},   "status":{"statusCategory":{"key":"new"}},          "labels":["dataset:TRENDED_REPORT"]}}
]'));

-------------------------------------------------------------------------------
-- 3. Recompute scores so the Active Issues dimension reflects the incidents,
--    then inspect.
-------------------------------------------------------------------------------
CALL SP_DTS_COMPUTE_SCORES();

SELECT REPORT_FAMILY, LAYER, SEVERITY, STATUS, DETAIL
FROM   DTS_OBSERVABILITY_INCIDENT
WHERE  INCIDENT_TYPE = 'JIRA' AND STATUS = 'OPEN'
ORDER BY REPORT_FAMILY, LAYER, SEVERITY;

## 11 · Run the pipeline (Track B — no grant)

Clones the 5 scored objects, computes DMF-equivalent metrics in plain SQL over the clones, then computes the trust scores. Each CALL is its own cell so you can inspect the returned VARIANT (`failed`/`failures`) between steps.

In [ ]:
CALL SP_DTS_CLONE_SCORED_OBJECTS();

In [ ]:
CALL SP_DTS_COMPUTE_DQ_MEASUREMENTS();

In [ ]:
CALL SP_DTS_COMPUTE_SCORES();

## 12 · Results — latest trust score per object

In [ ]:
SELECT REPORT_FAMILY, LAYER, DQ_MEASUREMENT_LAYER, TRUST_SCORE, TRUST_BAND,
       MEASURABLE_CEILING, FOUNDATIONAL_GAP_FLAG, FOUNDATIONAL_GAPS
FROM   DTS_VW_DATASET_TRUST_SCORE_LATEST
ORDER BY TRUST_SCORE DESC;

## 13 · Streamlit apps  _(manual step — not fully runnable here)_

`ddl/50` creates a stage + two `STREAMLIT` objects, but you must first **upload** `app/config.py`, `app/shared.py`, `app/trust_score_app.py`, `app/ai_use_cases_app.py`, and `app/.streamlit/config.toml` to the stage. Run these cells after the files are uploaded.

In [ ]:
-------------------------------------------------------------------------------
-- 50_create_streamlit_apps.sql  (v2)
--
-- Creates the TWO Streamlit-in-Snowflake (SiS) app objects in
-- EDLE_DW_DB.PNC_DATA so they show up as separate apps in Snowsight.
--
-- Each SiS app is identified by (database, schema, name). Two deploys with the
-- same triple replace each other, so we create two explicitly named STREAMLIT
-- objects sharing one stage but pointing at different MAIN_FILEs.
--
-- Run order:
--   1. Deploy the DTS_ objects first (ddl/00 -> 40) and run SP_DTS_COMPUTE_SCORES.
--   2. Run Section A once to create the stage.
--   3. Upload app/*.py (incl. config.py) + app/.streamlit/config.toml.
--   4. Run Section C to create the two STREAMLIT objects.
--   5. Run Section D to grant USAGE.
--   6. Run Section E to verify.
-------------------------------------------------------------------------------

USE ROLE      PNC_DEVELOPER_RL;

-------------------------------------------------------------------------------
-- SECTION A: ONE-TIME STAGE FOR APP SOURCE FILES
-------------------------------------------------------------------------------
CREATE STAGE IF NOT EXISTS EDLE_DW_DB.PNC_DATA.DTS_STREAMLIT_APP_STAGE
    DIRECTORY = (ENABLE = TRUE)
    COMMENT   = 'Source files for the v2 Data Trust Score Streamlit apps';

-------------------------------------------------------------------------------
-- SECTION B: FILE UPLOAD (SnowSQL example -- skip if using Snowsight UI)
--
-- Required files in the stage (flat, plus the .streamlit/ subfolder):
--   config.py            -- single source of truth (imported by shared.py)
--   shared.py            -- loaders / constants / CSS
--   trust_score_app.py   -- Trust Score app entry point
--   ai_use_cases_app.py  -- AI Use Cases app entry point
--   .streamlit/config.toml
--
-- Snowsight UI:
--   Data > Databases > EDLE_DW_DB > PNC_DATA > Stages >
--   DTS_STREAMLIT_APP_STAGE > +Files. Drag the files in (keep .streamlit/).
--
-- SnowSQL (from repo root):
--   snowsql -q "PUT file://app/config.py            @EDLE_DW_DB.PNC_DATA.DTS_STREAMLIT_APP_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE"
--   snowsql -q "PUT file://app/shared.py            @EDLE_DW_DB.PNC_DATA.DTS_STREAMLIT_APP_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE"
--   snowsql -q "PUT file://app/trust_score_app.py   @EDLE_DW_DB.PNC_DATA.DTS_STREAMLIT_APP_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE"
--   snowsql -q "PUT file://app/ai_use_cases_app.py  @EDLE_DW_DB.PNC_DATA.DTS_STREAMLIT_APP_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE"
--   snowsql -q "PUT file://app/.streamlit/config.toml @EDLE_DW_DB.PNC_DATA.DTS_STREAMLIT_APP_STAGE/.streamlit/ AUTO_COMPRESS=FALSE OVERWRITE=TRUE"

ALTER STAGE EDLE_DW_DB.PNC_DATA.DTS_STREAMLIT_APP_STAGE REFRESH;

LS @EDLE_DW_DB.PNC_DATA.DTS_STREAMLIT_APP_STAGE;

-------------------------------------------------------------------------------
-- SECTION C: CREATE THE TWO STREAMLIT OBJECTS
-------------------------------------------------------------------------------
CREATE OR REPLACE STREAMLIT EDLE_DW_DB.PNC_DATA.DTS_PNC_DATA_TRUST_SCORE
    ROOT_LOCATION   = '@EDLE_DW_DB.PNC_DATA.DTS_STREAMLIT_APP_STAGE'
    MAIN_FILE       = 'trust_score_app.py'
    QUERY_WAREHOUSE = PNC_WH
    TITLE           = 'P&C Data Trust Score (v2)'
    COMMENT         = 'v2 governance surface: Portfolio Scorecard, Dataset Detail, Lineage DQ (INT->DW->PUBL), Methodology';

CREATE OR REPLACE STREAMLIT EDLE_DW_DB.PNC_DATA.DTS_PNC_HR_AI_USE_CASES
    ROOT_LOCATION   = '@EDLE_DW_DB.PNC_DATA.DTS_STREAMLIT_APP_STAGE'
    MAIN_FILE       = 'ai_use_cases_app.py'
    QUERY_WAREHOUSE = PNC_WH
    TITLE           = 'P&C HR Analytics (AI use cases)'
    COMMENT         = 'Applied analytics: Workforce Shifts, TA Analytics (requires VW_WORKFORCE_* views)';

-------------------------------------------------------------------------------
-- SECTION D: GRANT USAGE (edit roles to match your consumers)
-------------------------------------------------------------------------------
-- GRANT USAGE ON STREAMLIT EDLE_DW_DB.PNC_DATA.DTS_PNC_DATA_TRUST_SCORE TO ROLE PNC_REPORTING_RL;
-- GRANT USAGE ON STREAMLIT EDLE_DW_DB.PNC_DATA.DTS_PNC_HR_AI_USE_CASES  TO ROLE PNC_REPORTING_RL;


-------------------------------------------------------------------------------
-- SECTION E: VERIFY
-------------------------------------------------------------------------------
SHOW STREAMLITS IN SCHEMA EDLE_DW_DB.PNC_DATA;

SELECT "name" AS app_name, "title" AS app_title,
       "query_warehouse" AS warehouse, "url_id" AS url_id, "comment" AS comment
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY "name";